# Does adaptation compose?
### Independently trained language and genre modules on ⟨Swedish, literary⟩

Companion notebook for the ELMA/SILD take-home. It runs the study end to end:
builds the two module corpora and the target split, trains one LoRA module per
factor on disjoint data, composes them on the intersection cell, and produces
every number reported in the paper.

**The cell.** Target ⟨sv, literary⟩ — the Swedish side of `opus_books`, which is
one novel in three translation pairs. The language module is trained on Swedish
legal text (`opus_dgt`); the genre module on literary text in the fifteen other
languages of `opus_books`. Neither module ever sees the target cell.

**Runtime.** About 3.5 h on one A100 for the two trainings and roughly one hour
of evaluation. Trained adapters are cached, so a re-run skips the training.


---

### How to read this notebook

Every code cell carries one of two banners.

**`══ PLUMBING ══`** — infrastructure. Micro-batching and OOM handling, mounting
and unmounting adapters, tokenising, subsampling hidden states, writing JSON.
None of it is part of the argument: any of these cells could be replaced by a
different implementation without changing a single claim. Ten cells.

**`── CORE ──`** — the study. Every design decision, every measurement and every
result lives in one of these. If a number appears in the paper, it is printed by
a CORE cell.

The split is deliberate. The plumbing exists because a 2.25B backbone on one GPU
forces it, not because anything conceptual depends on it.

---


## 1 · Configuration

In [1]:
# ── CORE ───────────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]  = "true"

R           = 16          # LoRA rank
ALPHA       = 32          # alpha/r = 2, so r varies capacity and not effective step size
LORA_LR     = 1e-4
SEQ_LEN     = 512
BATCH       = 16          # logical batch, in blocks, per optimisation step
MICRO_INI   = 8           # chunk actually sent to the GPU; halved on OOM
SEED        = 20260816
TARGETS     = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

BACKBONE    = "BSC-LT/salamandra-2b"
TARGET      = "sv"
LEGAL_PAIR  = "hr-sv"     # the only opus_dgt pair containing Swedish
N_VAL_BLK   = 96          # blocks held out of each module corpus, only to measure it
EVAL_EVERY  = 500         # steps between intermediate evaluations
SAVE_EVERY  = 500         # steps between adapter checkpoints

RAW  = "/content/raw"
WORK = "/content/work"
os.makedirs(RAW, exist_ok=True); os.makedirs(WORK, exist_ok=True)

# Checkpoints go to Drive when it is available: the genre module is a long run.
CKPT = WORK
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT = "/content/drive/MyDrive/elma_sv"
    os.makedirs(CKPT, exist_ok=True)
except Exception as e:
    print("no Drive; checkpoints stay in /content/work —", e)
print("checkpoints →", CKPT)

Mounted at /content/drive
checkpoints → /content/drive/MyDrive/elma_sv


## 2 · Environment

In [2]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
!pip -q install -U "transformers>=4.44" "peft>=0.11" "huggingface_hub>=0.24" pyarrow accelerate 2>&1 | tail -2

# Colab ships torchao 0.10.0. Recent peft calls `is_torchao_available()` while
# building a LoRA layer and *raises* on an old version instead of degrading, so
# `get_peft_model` dies with an ImportError even though nothing here quantises.
# We do not use torchao, so the fix is to remove it — before importing peft.
!pip uninstall -y -q torchao 2>&1 | tail -1
import gc, json, math, random, re, sys, time, glob
import numpy as np, torch
import pyarrow.parquet as pq
from collections import defaultdict, Counter
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "no GPU available"

# Recorded, not just printed: several results depend on library behaviour that
# has changed between releases, so the resolved versions are part of the result.
import pyarrow, transformers, peft, huggingface_hub, accelerate
VERSIONS = {"python": sys.version.split()[0], "torch": torch.__version__,
            "transformers": transformers.__version__, "peft": peft.__version__,
            "accelerate": accelerate.__version__,
            "huggingface_hub": huggingface_hub.__version__,
            "numpy": np.__version__, "pyarrow": pyarrow.__version__,
            "gpu": torch.cuda.get_device_name(0)}
for k, v in VERSIONS.items(): print(f"  {k:<16} {v}")
print(f"  {'GPU memory':<16} {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def free():
    gc.collect(); torch.cuda.empty_cache()
set_seed()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 138.2 MB/s eta 0:00:00
  python           3.12.13
  torch            2.11.0+cu128
  transformers     5.15.0
  peft             0.20.0
  accelerate       1.14.0
  huggingface_hub  1.27.0
  numpy            2.0.2
  pyarrow          25.0.1
  gpu              NVIDIA A100-SXM4-40GB
  GPU memory       39.5 GiB


## 3 · The corpora

One decision here is not cosmetic.

**Training blocks must be language-pure.** `opus_books` stores translation
*pairs*, so chunking rows as they come out of the parquet files yields blocks
that alternate language every segment, with the same sentence appearing twice.
That trains translation, not genre. We group by language, chunk *within* each
language, and shuffle only the resulting list of blocks.

In [3]:
# ── CORE ───────────────────────────────────────────────────────────────
t0 = time.time()
snapshot_download("Helsinki-NLP/opus_books", repo_type="dataset",
                  local_dir=f"{RAW}/opus_books", allow_patterns=["*.parquet"], max_workers=8)
snapshot_download("Helsinki-NLP/opus_dgt", repo_type="dataset",
                  local_dir=f"{RAW}/opus_dgt", allow_patterns=[f"{LEGAL_PAIR}/*.parquet"],
                  max_workers=4)
print(f"downloaded in {time.time()-t0:.0f}s")

def read_pair(corpus, pair):
    rows = []
    for f in sorted(glob.glob(f"{RAW}/{corpus}/{pair}/*.parquet")):
        rows += pq.read_table(f).column("translation").to_pylist()
    return rows

# ---- LANGUAGE module corpus: all of the Swedish legal text -------------------
LANG = {TARGET: [s for s in ((r.get(TARGET) or "") for r in read_pair("opus_dgt", LEGAL_PAIR))
                 if s.strip()]}
print(f"language · {TARGET}: {len(LANG[TARGET]):,} segments · "
      f"{sum(len(s.split()) for s in LANG[TARGET]):,} words")

# ---- GENRE module corpus: all of opus_books minus every pair containing sv ---
GENRE = defaultdict(list)
for pair in sorted(os.path.basename(p) for p in glob.glob(f"{RAW}/opus_books/*-*")):
    sides = pair.split("-")
    if TARGET in sides:
        continue                      # the whole pair, not just the sv column
    for r in read_pair("opus_books", pair):
        for lg in sides:
            s = r.get(lg) or ""
            if s.strip(): GENRE[lg].append(s)
GENRE = dict(GENRE)
tot_g = sum(sum(len(s.split()) for s in v) for v in GENRE.values())
print(f"\ngenre · {len(GENRE)} languages · {sum(len(v) for v in GENRE.values()):,} segments · "
      f"{tot_g:,} words (complete, not subsampled)")
for lg, v in sorted(GENRE.items(), key=lambda kv: -sum(len(s.split()) for s in kv[1])):
    w = sum(len(s.split()) for s in v)
    print(f"   {lg:>3} {w:>10,} ({100*w/tot_g:>5.1f} %)")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

downloaded in 7s
language · sv: 696,334 segments · 11,135,049 words

genre · 15 languages · 2,483,074 segments · 48,601,840 words (complete, not subsampled)
    en 11,040,385 ( 22.7 %)
    fr  7,833,960 ( 16.1 %)
    hu  7,712,731 ( 15.9 %)
    es  7,008,693 ( 14.4 %)
    de  4,875,359 ( 10.0 %)
    nl  3,657,410 (  7.5 %)
    it  3,375,669 (  6.9 %)
    ru  1,635,987 (  3.4 %)
    ca    290,158 (  0.6 %)
    no    276,640 (  0.6 %)
    fi    254,346 (  0.5 %)
    eo    203,029 (  0.4 %)
    pt    175,428 (  0.4 %)
    pl    141,467 (  0.3 %)
    el    120,578 (  0.2 %)


The target is split **by chapter**, sending all three copies of a chapter to the
same partition. Cutting by row or by pair would necessarily place the same
sentence on both sides of the split, because the three pairs are three
segmentations of one work.

In [4]:
# ── CORE ───────────────────────────────────────────────────────────────
ROMAN = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,
         "XI":11,"XII":12,"XIII":13,"XIV":14,"XV":15,"XVI":16,"XVII":17,"XVIII":18,"XIX":19}
CHAPTER_RX = re.compile(r"^\s*Kapitel\s+([IVX]+)\s*$")
PART = {**{c: "train" for c in [0] + list(range(1, 13))},
        **{c: "val"   for c in (13, 14)},
        **{c: "test"  for c in range(15, 20)}}
BOUNDARIES = (13, 15)          # the two that must be present in all three pairs

TGT, boundaries_ok = defaultdict(list), True
for pair in sorted(os.path.basename(x) for x in glob.glob(f"{RAW}/opus_books/*-*")):
    if TARGET not in pair.split("-"):
        continue
    chapter, seen = 0, set()
    for r in read_pair("opus_books", pair):
        s = r.get(TARGET) or ""
        m = CHAPTER_RX.match(s)
        if m:
            chapter = ROMAN[m.group(1)]; seen.add(chapter)
        if s.strip(): TGT[PART[chapter]].append(s)
    if not set(BOUNDARIES) <= seen: boundaries_ok = False

print(f"target · partition boundaries found in all three pairs: {boundaries_ok}")
for k in ("train", "val", "test"):
    print(f"   {k:>5}: {len(TGT[k]):>6,} segments · "
          f"{sum(len(s.split()) for s in TGT[k]):>7,} words")

target · partition boundaries found in all three pairs: True
   train:  5,849 segments · 128,246 words
     val:  1,184 segments ·  25,929 words
    test:  2,062 segments ·  45,430 words


## 4 · Backbone and blocks

In [5]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
tok = AutoTokenizer.from_pretrained(BACKBONE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BACKBONE, dtype=torch.bfloat16).to("cuda")
base.config.use_cache = False
for p in base.parameters(): p.requires_grad_(False)
FINGERPRINT0 = sum(float(p.detach().float().sum()) for p in base.parameters())
print(f"{sum(p.numel() for p in base.parameters())/1e9:.2f} B parameters "
      f"· fingerprint {FINGERPRINT0:.6e}")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/989 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 4.81MB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 37.0MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.51GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

2.25 B parameters · fingerprint 5.112613e+04


In [6]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
def tokenize(segments, join=500, batch=64):
    """Tokenise in batches so the fast tokeniser actually parallelises. Returns
    np.int32: with 100M tokens a Python list of ints would waste several GB."""
    parts  = []
    groups = ["\n".join(segments[j:j+join]) for j in range(0, len(segments), join)]
    for i in range(0, len(groups), batch):
        for e in tok(groups[i:i+batch], add_special_tokens=False)["input_ids"]:
            parts.append(np.asarray(e, dtype=np.int32))
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.int32)

In [7]:
# ── CORE ───────────────────────────────────────────────────────────────
def pure_blocks(d, label):
    """Chunk WITHIN each language; shuffle only the resulting list of blocks."""
    chunks, census, t0 = [], {}, time.time()
    for lg, segments in sorted(d.items()):
        ids = tokenize(segments)
        n = len(ids) // SEQ_LEN
        if n == 0: continue
        chunks.append(torch.from_numpy(ids[:n*SEQ_LEN].reshape(n, SEQ_LEN)))
        census[lg] = n
    X = torch.cat(chunks, 0)
    g = torch.Generator().manual_seed(SEED)
    X = X[torch.randperm(X.shape[0], generator=g)]
    print(f"{label}: {X.shape[0]:,} blocks · {X.numel():,} tokens · "
          f"{len(census)} languages · {time.time()-t0:.0f}s")
    return X

def flat_blocks(segments, label):
    ids = tokenize(segments); n = len(ids) // SEQ_LEN
    X = torch.from_numpy(ids[:n*SEQ_LEN].reshape(n, SEQ_LEN))
    print(f"{label}: {X.shape[0]:,} blocks · {X.numel():,} tokens")
    return X

X_L = pure_blocks(LANG,  "language")
X_G = pure_blocks(GENRE, "genre")
CORPUS = {"language": X_L[:-N_VAL_BLK], "genre": X_G[:-N_VAL_BLK]}

EVAL = {"language·val": X_L[-N_VAL_BLK:],
        "genre·val":    X_G[-N_VAL_BLK:],
        "target·val":   flat_blocks(TGT["val"],  "target·val"),
        "target·TEST":  flat_blocks(TGT["test"], "target·TEST")}
BYTES  = {k: sum(len(tok.decode(x[1:].tolist()).encode("utf-8")) for x in X)
          for k, X in EVAL.items()}
TOKENS = {k: int(X.shape[0]*(SEQ_LEN-1)) for k, X in EVAL.items()}

STEPS = {m: CORPUS[m].shape[0] // BATCH for m in CORPUS}      # exactly one pass
print()
for m in CORPUS:
    print(f"{m:>9}: {CORPUS[m].shape[0]:>7,} training blocks → {STEPS[m]:>6,} steps (1 pass)")
for k in EVAL:
    print(f"   {k:>13}: {EVAL[k].shape[0]:>5,} blocks · {BYTES[k]:>9,} bytes")

language: 40,430 blocks · 20,700,160 tokens · 1 languages · 11s
genre: 157,057 blocks · 80,413,184 tokens · 15 languages · 34s
target·val: 75 blocks · 38,400 tokens
target·TEST: 129 blocks · 66,048 tokens

 language:  40,334 training blocks →  2,520 steps (1 pass)
    genre: 156,961 training blocks →  9,810 steps (1 pass)
    language·val:    96 blocks ·   198,567 bytes
       genre·val:    96 blocks ·   188,104 bytes
      target·val:    75 blocks ·   153,689 bytes
     target·TEST:   129 blocks ·   268,963 bytes


## 5 · Training

Both metrics are reported. **Bits per byte** is primary because it does not
depend on how the tokeniser segments words; perplexity is printed alongside
because it is what readers expect. All systems here share a tokeniser, so the
two order identically — but only for that reason.

Each module gets **exactly one pass** over the whole of its corpus. No data is
shared between the two, and none comes from the target cell.

In [8]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
def fwd_bwd(m, xb, micro):
    """One optimisation step's worth of forward/backward, split into micro-batches."""
    while True:
        try:
            total = 0.0
            for i in range(0, xb.shape[0], micro):
                x = xb[i:i+micro].to("cuda", non_blocking=True).long()
                out = m(input_ids=x, labels=x)
                (out.loss * x.shape[0] / xb.shape[0]).backward()
                total += float(out.loss.detach()) * x.shape[0]
                del out, x
            return total/xb.shape[0], micro
        except torch.cuda.OutOfMemoryError:
            m.zero_grad(set_to_none=True); free()
            micro = max(1, micro//2)
            print(f"      OOM → micro={micro}")
            if micro == 1: raise

def extract_deltas(pm):
    """Pull (B, A, scaling) out of a PEFT model, one entry per adapted matrix."""
    d = {}
    for n, mod in pm.named_modules():
        if hasattr(mod, "lora_A") and "default" in mod.lora_A:
            A = mod.lora_A["default"].weight.detach().float().cpu()
            B = mod.lora_B["default"].weight.detach().float().cpu()
            d[n.replace("base_model.model.", "").replace(".base_layer", "")] = \
                (B, A, float(mod.scaling["default"]))
    return d

In [9]:
# ── CORE ───────────────────────────────────────────────────────────────
@torch.no_grad()
def measure(m, key, micro=4):
    """Returns (bits per byte, perplexity) on evaluation set `key`."""
    X, nb, nt = EVAL[key], BYTES[key], TOKENS[key]
    m.eval(); nats, i = 0.0, 0
    while i < X.shape[0]:
        try:
            x = X[i:i+micro].to("cuda").long()
            lg = m(input_ids=x).logits[:, :-1].float(); y = x[:, 1:]
            nats += float(torch.nn.functional.cross_entropy(
                lg.reshape(-1, lg.shape[-1]), y.reshape(-1), reduction="sum"))
            del lg, x, y; i += micro
        except torch.cuda.OutOfMemoryError:
            free(); micro = max(1, micro//2)
            print(f"      OOM during eval → micro={micro}")
            if micro == 1: raise
    m.train()
    return nats/math.log(2)/nb, math.exp(nats/nt)

def train(module, watch):
    """One full pass over CORPUS[module]. `watch` lists the sets to evaluate."""
    global base
    path = f"{CKPT}/tau_{module}_r{R}.pt"
    if os.path.exists(path):
        print(f"  {path} already exists → reusing it (delete the file to retrain)")
        return torch.load(path, map_location="cpu", weights_only=False), {}
    set_seed()
    X, steps = CORPUS[module], STEPS[module]
    cfg = LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=0.0, bias="none",
                     target_modules=TARGETS, task_type="CAUSAL_LM")
    pm = get_peft_model(base, cfg)
    for p in pm.parameters():
        if p.requires_grad: p.data = p.data.float()
    trainable = [p for p in pm.parameters() if p.requires_grad]
    print(f"  {sum(p.numel() for p in trainable):,} trainable parameters · {steps:,} steps")
    opt = torch.optim.AdamW(trainable, lr=LORA_LR, weight_decay=0.0, betas=(0.9, 0.95))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LORA_LR, total_steps=steps,
                                                pct_start=0.03, anneal_strategy="cos")
    g = torch.Generator().manual_seed(SEED)
    order = torch.randperm(X.shape[0], generator=g)
    micro, hist, t0, acc = MICRO_INI, {}, time.time(), []
    for step in range(1, steps+1):
        idx = order[(step-1)*BATCH : step*BATCH]
        opt.zero_grad(set_to_none=True)
        loss, micro = fwd_bwd(pm, X[idx], micro)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        opt.step(); sched.step(); acc.append(loss)
        if step % EVAL_EVERY == 0 or step == steps:
            ev = {k: measure(pm, k) for k in watch}
            hist[step] = {"loss": float(np.mean(acc[-EVAL_EVERY:])), **ev}
            elapsed = time.time()-t0
            print(f"    step {step:>6}/{steps} · loss {np.mean(acc[-EVAL_EVERY:]):.4f} · " +
                  " · ".join(f"{k} {v[0]:.4f}bpb/{v[1]:.2f}ppl" for k, v in ev.items()) +
                  f" · {elapsed/60:.1f}min (~{elapsed/step*(steps-step)/60:.0f}min left)")
        if step % SAVE_EVERY == 0:
            torch.save(extract_deltas(pm), f"{CKPT}/tau_{module}_r{R}.partial.pt")
    d = extract_deltas(pm)
    torch.save(d, path)
    try: base = pm.unload()
    except AttributeError: base = pm.base_model.unload()
    del pm, opt, sched, trainable; free()
    h = sum(float(p.detach().float().sum()) for p in base.parameters())
    assert abs(h-FINGERPRINT0) < 1e-3*max(1.0, abs(FINGERPRINT0)), "the backbone moved!"
    print(f"  saved to {path}")
    return d, hist

In [10]:
# ── CORE ───────────────────────────────────────────────────────────────
def ensure_clean_backbone():
    """A `get_peft_model` call that raises part-way leaves LoRA layers injected
    in `base` and does not undo them. They are zero-initialised, so nothing
    looks wrong — but a backbone measurement taken on top of them is not the
    backbone's, and every Δ computed against it is off. This happened once,
    when an unrelated ImportError killed the training cell mid-injection.
    `check_base()` below cannot catch it, because it is installed later and
    takes its snapshot from whatever state it finds."""
    global base
    stale = [n for n, m in base.named_modules() if hasattr(m, "lora_A")]
    if not stale:
        return
    print(f"{len(stale)} stale PEFT layers found in the backbone; reloading "
          f"from the Hub (cached, quick)")
    del base; gc.collect(); torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(BACKBONE, dtype=torch.bfloat16).to("cuda")
    base.config.use_cache = False
    for p in base.parameters(): p.requires_grad_(False)
    assert not [n for n, m in base.named_modules() if hasattr(m, "lora_A")]

ensure_clean_backbone()
BEFORE = {k: measure(base, k) for k in EVAL}
print("untouched backbone:")
for k, v in BEFORE.items():
    print(f"   {k:>14}: {v[0]:.4f} bpb · {v[1]:8.2f} ppl")

print("\n=== LANGUAGE module  ⟨sv, ¬literary⟩ ===")
TAU_L, HIST_L = train("language", ["language·val", "target·val"])
print("\n=== GENRE module  ⟨L∖sv, literary⟩ ===")
TAU_G, HIST_G = train("genre", ["genre·val", "target·val"])
print(f"\nτ_L: {len(TAU_L)} matrices · τ_G: {len(TAU_G)} matrices")

untouched backbone:
     language·val: 0.6991 bpb ·     7.11 ppl
        genre·val: 1.0363 bpb ·    15.71 ppl
       target·val: 1.1396 bpb ·    23.75 ppl
      target·TEST: 1.1045 bpb ·    22.73 ppl

=== LANGUAGE module  ⟨sv, ¬literary⟩ ===
  /content/drive/MyDrive/elma_sv/tau_language_r16.pt already exists → reusing it (delete the file to retrain)

=== GENRE module  ⟨L∖sv, literary⟩ ===
  /content/drive/MyDrive/elma_sv/tau_genre_r16.pt already exists → reusing it (delete the file to retrain)

τ_L: 168 matrices · τ_G: 168 matrices


## 6 · How much did each module learn on its own domain?

Summing two LoRA modules is free and exact: since
`ΔW₁ + ΔW₂ = [s₁B₁ s₂B₂]·[A₁;A₂]`, the sum is again a single adapter of rank at
most `2r`. The projection composition `θ₀ + ΔW_L + (I − U_L U_Lᵀ)ΔW_G` keeps that
property, because `P⊥ΔW_G = (P⊥B_G)·A_G` is still a rank-`r` adapter — so it is
merged and served at exactly the same cost as any other.

In [11]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
@torch.no_grad()
def load_and_measure(adapter, keys):
    """Mount a prebuilt adapter, measure, unmount. Never touches the base weights."""
    global base
    if adapter is None:
        return {k: measure(base, k) for k in keys}
    R2 = max(B.shape[1] for B, _ in adapter.values())
    cfg = LoraConfig(r=R2, lora_alpha=R2, lora_dropout=0.0, bias="none",   # alpha/r = 1
                     target_modules=TARGETS, task_type="CAUSAL_LM")
    pm = get_peft_model(base, cfg); placed = 0
    for n, mod in pm.named_modules():
        if not (hasattr(mod, "lora_A") and "default" in mod.lora_A): continue
        name = n.replace("base_model.model.", "").replace(".base_layer", "")
        if name not in adapter: continue
        B, A = adapter[name]
        dt = mod.lora_A["default"].weight.dtype
        mod.lora_A["default"].weight.data = A.to("cuda", dt)
        mod.lora_B["default"].weight.data = B.to("cuda", dt)
        placed += 1
    assert placed == len(adapter), f"placed {placed} of {len(adapter)}"
    v = {k: measure(pm, k) for k in keys}
    try: base = pm.unload()
    except AttributeError: base = pm.base_model.unload()
    del pm; free()
    h = sum(float(p.detach().float().sum()) for p in base.parameters())
    assert abs(h-FINGERPRINT0) < 1e-3*max(1.0, abs(FINGERPRINT0)), "the backbone moved!"
    return v

In [12]:
# ── CORE ───────────────────────────────────────────────────────────────
def project_out(B_ref, B_target):
    """Remove from B_target's column space whatever already lies in B_ref's."""
    Q, _ = torch.linalg.qr(B_ref.float()); Bo = B_target.float()
    return Bo - Q @ (Q.T @ Bo)

def combine(pieces):
    """Summing two LoRAs is concatenating their factors: [s_L B_L , s_G B_G]·[A_L ; A_G].
    Each piece is (tau, lambda, project_against_or_None)."""
    out, names = {}, sorted(set.intersection(*[set(t) for t, _, _ in pieces]))
    for n in names:
        Bs, As = [], []
        for tau, lam, against in pieces:
            B, A, s = tau[n]; B = B.float()*(s*lam)
            if against is not None: B = project_out(against[n][0], B)
            Bs.append(B); As.append(A.float())
        out[n] = (torch.cat(Bs, 1), torch.cat(As, 0))
    return out

# sanity check: concatenating really is summing
n0 = sorted(TAU_L)[0]
BL, AL, sL = TAU_L[n0]; BG, AG, sG = TAU_G[n0]
direct = sL*(BL.float() @ AL.float()) + sG*(BG.float() @ AG.float())
Bc, Ac = combine([(TAU_L, 1.0, None), (TAU_G, 1.0, None)])[n0]
print(f"check · concatenating == summing: max error {float((Bc@Ac-direct).abs().max()):.2e} "
      f"against ‖ΔW‖∞ = {float(direct.abs().max()):.4f}")

check · concatenating == summing: max error 1.40e-09 against ‖ΔW‖∞ = 0.0057


In [13]:
# ── CORE ───────────────────────────────────────────────────────────────
ONLY_L = load_and_measure(combine([(TAU_L, 1.0, None)]), list(EVAL))
ONLY_G = load_and_measure(combine([(TAU_G, 1.0, None)]), list(EVAL))

print(f"\n{'module on its own held-out data':<36}{'bpb before':>11}{'bpb after':>11}"
      f"{'Δ':>10}{'%':>8}{'ppl before':>12}{'ppl after':>11}")
print("-" * 99)
for name, res, key in (("τ_L  language ⟨sv, ¬lit⟩",  ONLY_L, "language·val"),
                       ("τ_G  genre ⟨L∖sv, lit⟩",    ONLY_G, "genre·val")):
    a, d = BEFORE[key], res[key]
    print(f"{name:<36}{a[0]:>11.4f}{d[0]:>11.4f}{d[0]-a[0]:>+10.4f}"
          f"{100*(d[0]-a[0])/a[0]:>7.1f}%{a[1]:>12.2f}{d[1]:>11.2f}")

print("\neach module on the OTHER's domain and on the target, for reference:")
for name, res in (("τ_L", ONLY_L), ("τ_G", ONLY_G)):
    print(f"  {name}: " + " · ".join(
        f"{k} {res[k][0]:.4f} ({res[k][0]-BEFORE[k][0]:+.4f})" for k in EVAL))


module on its own held-out data      bpb before  bpb after         Δ       %  ppl before  ppl after
---------------------------------------------------------------------------------------------------
τ_L  language ⟨sv, ¬lit⟩                 0.6991     0.5942   -0.1049  -15.0%        7.11       5.30
τ_G  genre ⟨L∖sv, lit⟩                   1.0363     0.9048   -0.1315  -12.7%       15.71      11.08

each module on the OTHER's domain and on the target, for reference:
  τ_L: language·val 0.5942 (-0.1049) · genre·val 1.0699 (+0.0336) · target·val 1.1637 (+0.0240) · target·TEST 1.1282 (+0.0237)
  τ_G: language·val 0.7921 (+0.0930) · genre·val 0.9048 (-0.1315) · target·val 1.1633 (+0.0236) · target·TEST 1.1279 (+0.0234)


## 7 · The target test

Seven systems on `target·TEST`, which nothing above has touched: the backbone,
each module alone, their plain **sum**, their uniform **average** as the standard
weight-space soup, and the **projection** in both orders — which module stays
intact is a real choice, and the asymmetry is itself a result.

In [14]:
# ── CORE ───────────────────────────────────────────────────────────────
SYSTEMS = [
    ("θ₀   backbone",            None),
    ("τ_L  language only",       [(TAU_L, 1.0, None)]),
    ("τ_G  genre only",          [(TAU_G, 1.0, None)]),
    ("SUM  τ_L + τ_G",           [(TAU_L, 1.0, None), (TAU_G, 1.0, None)]),
    ("MEAN ½(τ_L + τ_G)",        [(TAU_L, 0.5, None), (TAU_G, 0.5, None)]),
    ("PROJ τ_L + P⊥τ_G",         [(TAU_L, 1.0, None), (TAU_G, 1.0, TAU_L)]),
    ("PROJ τ_G + P⊥τ_L",         [(TAU_G, 1.0, None), (TAU_L, 1.0, TAU_G)]),
]

RES = {}
for label, recipe in SYSTEMS:
    RES[label] = (BEFORE if recipe is None else
                  load_and_measure(combine(recipe), list(EVAL)))
    print(f"  {label:<22} test {RES[label]['target·TEST'][0]:.4f} bpb")

b0 = RES["θ₀   backbone"]
print(f"\n{'system':<22}{'bpb TEST':>11}{'Δ vs θ₀':>10}{'ppl TEST':>11}"
      f"{'bpb val':>10}{'language':>10}{'genre':>10}")
print("-" * 84)
for label, _ in SYSTEMS:
    r = RES[label]
    print(f"{label:<22}{r['target·TEST'][0]:>11.4f}"
          f"{r['target·TEST'][0]-b0['target·TEST'][0]:>+10.4f}"
          f"{r['target·TEST'][1]:>11.2f}{r['target·val'][0]:>10.4f}"
          f"{r['language·val'][0]:>10.4f}{r['genre·val'][0]:>10.4f}")

t = "target·TEST"
s_sum  = RES["SUM  τ_L + τ_G"][t][0]
s_proj = min(RES["PROJ τ_L + P⊥τ_G"][t][0], RES["PROJ τ_G + P⊥τ_L"][t][0])
best   = min(RES, key=lambda k: RES[k][t][0])
print(f"\nbest system on the test split: «{best}» ({RES[best][t][0]:.4f} bpb)")
print(f"projection − sum: {s_proj-s_sum:+.4f} bpb → "
      f"{'the projection BEATS the sum' if s_proj < s_sum else 'the projection does NOT beat the sum'}")
print(f"does any composition beat the backbone? "
      f"{'yes, ' + best if RES[best][t][0] < b0[t][0] else 'no'}")

  θ₀   backbone          test 1.1045 bpb
  τ_L  language only     test 1.1282 bpb
  τ_G  genre only        test 1.1279 bpb
  SUM  τ_L + τ_G         test 1.1424 bpb
  MEAN ½(τ_L + τ_G)      test 1.0942 bpb
  PROJ τ_L + P⊥τ_G       test 1.1411 bpb
  PROJ τ_G + P⊥τ_L       test 1.1415 bpb

system                   bpb TEST   Δ vs θ₀   ppl TEST   bpb val  language     genre
------------------------------------------------------------------------------------
θ₀   backbone              1.1045   +0.0000      22.73    1.1396    0.6991    1.0363
τ_L  language only         1.1282   +0.0237      24.31    1.1637    0.5942    1.0699
τ_G  genre only            1.1279   +0.0234      24.29    1.1633    0.7921    0.9048
SUM  τ_L + τ_G             1.1424   +0.0379      25.30    1.1778    0.6572    0.9336
MEAN ½(τ_L + τ_G)          1.0942   -0.0104      22.08    1.1264    0.6381    0.9531
PROJ τ_L + P⊥τ_G           1.1411   +0.0366      25.21    1.1761    0.6549    0.9322
PROJ τ_G + P⊥τ_L           1.141

## 8 · Where the two modules overlap in parameter space

For each adapted matrix, the singular values of `U_Lᵀ U_G` are the cosines of the
principal angles between the two modules' column spaces. The control is the same
quantity between two *random* rank-`r` subspaces of the same dimension, which is
not zero but roughly `√(r/d)`, so everything is reported as a multiple of chance.

If the `×chance` and `removed` columns come out ordered alike, the projection
operator targets itself: it removes exactly where the two modules write into the
same directions, without being told where that is.

In [15]:
# ── CORE ───────────────────────────────────────────────────────────────
def kind(n):
    for t_ in TARGETS:
        if t_ in n: return t_
    return n.split(".")[-1]

angles, surviving = defaultdict(list), defaultdict(lambda: [0.0, 0.0])
for n in sorted(set(TAU_L) & set(TAU_G)):
    BL, AL, sL = TAU_L[n]; BG, AG, sG = TAU_G[n]
    QL, _ = torch.linalg.qr(BL.float()); QG, _ = torch.linalg.qr(BG.float())
    cos = torch.linalg.svdvals(QL[:, :R].T @ QG[:, :R])
    d = BL.shape[0]
    g = torch.Generator().manual_seed(SEED + d + R)
    Z1, _ = torch.linalg.qr(torch.randn(d, R, generator=g))
    Z2, _ = torch.linalg.qr(torch.randn(d, R, generator=g))
    angles[kind(n)].append((cos, torch.linalg.svdvals(Z1.T @ Z2)))
    dG  = sG*(BG.float() @ AG.float())
    dGp = project_out(BL.float(), sG*BG.float()) @ AG.float()
    s_ = surviving[kind(n)]
    s_[0] += float(dG.norm())**2; s_[1] += float(dGp.norm())**2

print(f"{'projection':>11} {'mean cos':>10} {'chance':>8} {'×chance':>8} {'max':>8}"
      f" {'‖ΔW_G‖ survives':>17} {'removed':>9}")
print("-" * 76)
for t_ in TARGETS:
    if t_ not in angles: continue
    c = torch.stack([a for a, _ in angles[t_]]); z = torch.stack([b for _, b in angles[t_]])
    f = math.sqrt(surviving[t_][1]/surviving[t_][0])
    print(f"{t_:>11} {float(c.mean()):>10.4f} {float(z.mean()):>8.4f} "
          f"{float(c.mean())/float(z.mean()):>7.2f}× {float(c.max()):>8.4f}"
          f" {100*f:>16.1f} % {100*(1-f):>8.1f} %")

 projection   mean cos   chance  ×chance      max   ‖ΔW_G‖ survives   removed
----------------------------------------------------------------------------
     q_proj     0.1462   0.0772    1.89×   0.7384             96.2 %      3.8 %
     k_proj     0.1460   0.0772    1.89×   0.7896             97.0 %      3.0 %
     v_proj     0.0861   0.0772    1.12×   0.2536             99.5 %      0.5 %
     o_proj     0.0811   0.0772    1.05×   0.2501             99.5 %      0.5 %
  gate_proj     0.0474   0.0449    1.06×   0.1142             99.8 %      0.2 %
    up_proj     0.0474   0.0449    1.05×   0.1168             99.8 %      0.2 %
  down_proj     0.0847   0.0772    1.10×   0.2051             99.5 %      0.5 %


## 9 · A guard against a contaminated backbone

Every cell below mounts an adapter, measures, and unmounts it. If one of them
dies in between, the backbone keeps whatever was merged into it and everything
measured afterwards is wrong, silently.

The obvious guard — comparing the sum of all 2.25B parameters — does not work.
Merging a composition moves that sum far less than any usable tolerance, because
the ΔW have near-zero mean and the signs cancel: it detects a catastrophe, not a
contamination. `check_base()` keeps an exact copy of a sample of matrices and
compares them bit by bit, and the cell verifies that the guard itself fires on a
perturbation of 1e-2 in a single weight.

In [16]:
# ── CORE ───────────────────────────────────────────────────────────────
def _layer_of(n):
    for p in n.split("."):
        if p.isdigit(): return int(p)
    return -1

_all_w = [n for n, _ in base.named_parameters()
          if any(t in n for t in TARGETS) and n.endswith(".weight")]
_layers = sorted({_layer_of(n) for n in _all_w})
_sample = [n for n in _all_w
           if _layer_of(n) in (_layers[0], _layers[len(_layers)//2], _layers[-1])]
WITNESS = {n: p.detach().clone() for n, p in base.named_parameters() if n in _sample}
print(f"witness matrices stored: {len(WITNESS)} from layers "
      f"{_layers[0]}, {_layers[len(_layers)//2]} and {_layers[-1]} "
      f"({sum(v.numel() for v in WITNESS.values())*2/2**20:.0f} MB)")

def check_base(where=""):
    """Fail if ANY bit of the witness matrices has changed."""
    bad = [n for n, p in base.named_parameters()
           if n in WITNESS and not torch.equal(p.detach(), WITNESS[n])]
    assert not bad, (f"the backbone is contaminated{' at ' + where if where else ''}: "
                     f"{len(bad)} of {len(WITNESS)} witness matrices changed, e.g. {bad[0]}")
    return True

# A guard that has never fired is not a guard. Perturb one weight and check.
_n0 = next(iter(WITNESS)); _p0 = dict(base.named_parameters())[_n0]
_orig = _p0.data.flatten()[0].clone()
_p0.data.view(-1)[0] += 1e-2
try:
    check_base("selftest")
    print("WARNING: the guard missed a perturbation — fix it before continuing")
except AssertionError:
    print("guard self-test: a 1e-2 perturbation in a single weight is detected ✓")
finally:
    _p0.data.view(-1)[0] = _orig
check_base("after the self-test")
del _p0, _orig, _n0

witness matrices stored: 21 from layers 0, 12 and 23 (287 MB)
guard self-test: a 1e-2 perturbation in a single weight is detected ✓


## 10 · The statistical test: resampling chapters, not blocks

The resampling unit is not arbitrary. The target is **one work in three
segmentations**, so two blocks of the same chapter coming from two different
pairs are practically the same text. Resampling blocks treats three copies as
three observations and narrows the interval on an independence that does not
exist. The honest unit is the **chapter**.

Both are computed and printed side by side, because the difference between them
*is* the argument. With five chapters the percentile bootstrap draws from only
126 distinct resamples, so the interval is essentially the range of the five
chapter differences — which is why the per-chapter values are printed too.

In [17]:
# ── CORE ───────────────────────────────────────────────────────────────
print("building the test split chunked BY CHAPTER…")
target_pairs = [p for p in sorted(os.path.basename(x)
                                  for x in glob.glob(f"{RAW}/opus_books/*-*"))
                if TARGET in p.split("-")]
by_chapter = defaultdict(list)
for pair in target_pairs:
    chapter = 0
    for r in read_pair("opus_books", pair):
        s = r.get(TARGET) or ""
        m = CHAPTER_RX.match(s)
        if m: chapter = ROMAN[m.group(1)]
        if PART[chapter] == "test" and s.strip():
            by_chapter[chapter].append(s)

BLK, CHAPLAB = [], []
for c in sorted(by_chapter):
    X = flat_blocks(by_chapter[c], f"  chapter {c}")
    if X.shape[0] == 0:
        print(f"  chapter {c}: under one block, dropped"); continue
    BLK.append(X); CHAPLAB += [c] * X.shape[0]
BLK = torch.cat(BLK, 0); CHAPLAB = np.array(CHAPLAB)
BYTES_BLK = np.array([len(tok.decode(x[1:].tolist()).encode("utf-8")) for x in BLK])
chapters = sorted(set(CHAPLAB.tolist()))
print(f"\n{BLK.shape[0]} blocks · {len(chapters)} chapters {chapters} · "
      f"{BYTES_BLK.sum():,} bytes")
print(f"(the notebook's TEST split had {EVAL['target·TEST'].shape[0]} blocks; "
      f"chunking each chapter separately loses its tail)")

building the test split chunked BY CHAPTER…
  chapter 15: 43 blocks · 22,016 tokens
  chapter 16: 11 blocks · 5,632 tokens
  chapter 17: 24 blocks · 12,288 tokens
  chapter 18: 21 blocks · 10,752 tokens
  chapter 19: 27 blocks · 13,824 tokens

126 blocks · 5 chapters [15, 16, 17, 18, 19] · 262,765 bytes
(the notebook's TEST split had 129 blocks; chunking each chapter separately loses its tail)


In [18]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
@torch.no_grad()
def nats_per_block(m, X, micro=4):
    """NLL in nats of every block separately."""
    m.eval(); out, i = [], 0
    while i < X.shape[0]:
        try:
            x = X[i:i+micro].to("cuda").long()
            lg = m(input_ids=x).logits[:, :-1].float(); y = x[:, 1:]
            l = torch.nn.functional.cross_entropy(
                lg.reshape(-1, lg.shape[-1]), y.reshape(-1),
                reduction="none").view(y.shape).sum(1)
            out += l.tolist(); del lg, x, y, l; i += micro
        except torch.cuda.OutOfMemoryError:
            free(); micro = max(1, micro//2)
            if micro == 1: raise
    m.train()
    return np.array(out)

@torch.no_grad()
def nats_of_system(adapter):
    global base
    if adapter is None:
        return nats_per_block(base, BLK)
    R2 = max(B.shape[1] for B, _ in adapter.values())
    cfg = LoraConfig(r=R2, lora_alpha=R2, lora_dropout=0.0, bias="none",
                     target_modules=TARGETS, task_type="CAUSAL_LM")
    pm = get_peft_model(base, cfg)
    for n, mod in pm.named_modules():
        if not (hasattr(mod, "lora_A") and "default" in mod.lora_A): continue
        name = n.replace("base_model.model.", "").replace(".base_layer", "")
        if name not in adapter: continue
        B_, A_ = adapter[name]; dt = mod.lora_A["default"].weight.dtype
        mod.lora_A["default"].weight.data = A_.to("cuda", dt)
        mod.lora_B["default"].weight.data = B_.to("cuda", dt)
    v = nats_per_block(pm, BLK)
    try: base = pm.unload()
    except AttributeError: base = pm.base_model.unload()
    del pm; free()
    check_base("nats_of_system")
    return v

NATS = {}
for label, recipe in SYSTEMS:
    NATS[label] = nats_of_system(None if recipe is None else combine(recipe))
    print(f"  {label:<22} bpb {NATS[label].sum()/math.log(2)/BYTES_BLK.sum():.4f}")

  θ₀   backbone          bpb 1.1050
  τ_L  language only     bpb 1.1286
  τ_G  genre only        bpb 1.1278
  SUM  τ_L + τ_G         bpb 1.1429
  MEAN ½(τ_L + τ_G)      bpb 1.0944
  PROJ τ_L + P⊥τ_G       bpb 1.1416
  PROJ τ_G + P⊥τ_L       bpb 1.1420


In [19]:
# ── CORE ───────────────────────────────────────────────────────────────
def ci(a, b, unit, B=10000):
    """95 % interval for bpb(a) − bpb(b), resampling `unit` with replacement.
    unit = 'chapter' groups blocks by chapter (correct).
    unit = 'block'   treats them as independent (incorrect, shown for contrast)."""
    rng = np.random.default_rng(SEED)
    groups = ([np.where(CHAPLAB == c)[0] for c in chapters] if unit == "chapter"
              else [np.array([i]) for i in range(len(BLK))])
    n, d = len(groups), np.empty(B)
    for k in range(B):
        sel = np.concatenate([groups[j] for j in rng.integers(0, n, n)])
        d[k] = (a[sel].sum() - b[sel].sum()) / math.log(2) / BYTES_BLK[sel].sum()
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

nb0 = NATS["θ₀   backbone"]
print(f"{'system':<22}{'Δbpb':>9}   {'95% CI by CHAPTER':>24}   {'95% CI by block':>22}")
print("-" * 84)
for label, _ in SYSTEMS:
    if label.startswith("θ₀"): continue
    m1, lo1, hi1 = ci(NATS[label], nb0, "chapter")
    m2, lo2, hi2 = ci(NATS[label], nb0, "block")
    star = " *" if not (lo1 <= 0 <= hi1) else "  "
    print(f"{label:<22}{m1:>+9.4f}   [{lo1:+.4f}, {hi1:+.4f}]{star}   [{lo2:+.4f}, {hi2:+.4f}]")
print("\n* = the chapter-level interval excludes zero")
print(f"With {len(chapters)} chapters the correct interval is MUCH wider than the")
print("block-level one. That difference is not a defect of the method: it is the")
print("cell's real sample size.")

print(f"\nΔbpb against θ₀, chapter by chapter (is the sign stable?)")
print(f"{'system':<22}" + "".join(f"{'ch '+str(c):>10}" for c in chapters) + f"{'in favour':>11}")
print("-" * (22 + 10*len(chapters) + 11))
for label, _ in SYSTEMS:
    if label.startswith("θ₀"): continue
    row, wins = f"{label:<22}", 0
    for c in chapters:
        s = CHAPLAB == c
        d = (NATS[label][s].sum() - nb0[s].sum()) / math.log(2) / BYTES_BLK[s].sum()
        row += f"{d:>+10.4f}"; wins += d < 0
    print(row + f"{wins:>8}/{len(chapters)}")
print(f"\nA sign test with {len(chapters)} chapters gives p = 0.0312 at best: weak,")
print("but sign stability says more than the interval alone.")

for a, b in (("MEAN ½(τ_L + τ_G)", "SUM  τ_L + τ_G"),
             ("PROJ τ_L + P⊥τ_G",  "SUM  τ_L + τ_G"),
             ("MEAN ½(τ_L + τ_G)", "τ_G  genre only")):
    m, lo, hi = ci(NATS[a], NATS[b], "chapter")
    print(f"\n{a} − {b}: {m:+.4f}  CI95 [{lo:+.4f}, {hi:+.4f}]"
          f"{'  ✱ excludes zero' if not (lo <= 0 <= hi) else ''}")

system                     Δbpb          95% CI by CHAPTER          95% CI by block
------------------------------------------------------------------------------------
τ_L  language only      +0.0237   [+0.0217, +0.0257] *   [+0.0217, +0.0254]
τ_G  genre only         +0.0225   [+0.0167, +0.0271] *   [+0.0203, +0.0253]
SUM  τ_L + τ_G          +0.0378   [+0.0323, +0.0409] *   [+0.0351, +0.0407]
MEAN ½(τ_L + τ_G)       -0.0108   [-0.0150, -0.0077] *   [-0.0124, -0.0089]
PROJ τ_L + P⊥τ_G        +0.0365   [+0.0313, +0.0393] *   [+0.0337, +0.0393]
PROJ τ_G + P⊥τ_L        +0.0370   [+0.0317, +0.0401] *   [+0.0342, +0.0397]

* = the chapter-level interval excludes zero
With 5 chapters the correct interval is MUCH wider than the
block-level one. That difference is not a defect of the method: it is the
cell's real sample size.

Δbpb against θ₀, chapter by chapter (is the sign stable?)
system                     ch 15     ch 16     ch 17     ch 18     ch 19  in favour
---------------------------

## 11 · Lightweight supervised adaptation: two scalars

The second evaluation setting allows adapting the composed model on at most 1M
words of the target's training split. We spend that budget on **two scalars**,

$$\theta_0 + \lambda_L\,\Delta W_L + \lambda_G\,\Delta W_G$$

against the 14,917,632 of a fresh rank-16 adapter. The choice is not cosmetic:
with 187k training tokens a fresh adapter has roughly 80 parameters per token —
enough to learn the domain by itself, and therefore enough to erase the very
advantage the experiment is meant to detect.

Both λ are selected on the **whole training split** of the target, and the chosen
point is reported on val and test. The test surface is printed too, but *after*
the selection and only to measure what choosing on train costs.

Implementation: one adapter of rank `2R` is mounted once. `A` is the fixed
concatenation of both `A` factors; each grid point only rewrites
`B = [λ_L·s_L·B_L , λ_G·s_G·B_G]`.

In [20]:
# ── CORE ───────────────────────────────────────────────────────────────
STEP_L = 0.2
REFINE = True          # after the coarse grid, refine to 0.1 in the 3×3 neighbourhood

LAMS = [round(k*STEP_L, 3) for k in range(int(round(1/STEP_L))+1)]
print(f"grid {len(LAMS)}×{len(LAMS)} = {len(LAMS)**2} combinations · λ ∈ {LAMS}")

X_SEL = flat_blocks(TGT["train"], "target·train (complete)")
B_SEL = sum(len(tok.decode(x[1:].tolist()).encode("utf-8")) for x in X_SEL)
print(f"selection: {X_SEL.shape[0]} blocks · {X_SEL.shape[0]*SEQ_LEN:,} tokens · "
      f"{B_SEL:,} bytes → 100 % of the train split")

grid 6×6 = 36 combinations · λ ∈ [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
target·train (complete): 366 blocks · 187,392 tokens
selection: 366 blocks · 187,392 tokens · 752,787 bytes → 100 % of the train split


In [21]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
shared = sorted(set(TAU_L) & set(TAU_G))
PIECES = {}
for n in shared:
    BL, AL, sL = TAU_L[n]; BG, AG, sG = TAU_G[n]
    PIECES[n] = (BL.float()*sL, BG.float()*sG,
                 torch.cat([AL.float(), AG.float()], 0))
R2 = 2*R
cfg = LoraConfig(r=R2, lora_alpha=R2, lora_dropout=0.0, bias="none",   # alpha/r = 1
                 target_modules=TARGETS, task_type="CAUSAL_LM")
pm = get_peft_model(base, cfg)
MODS = {}
for n, mod in pm.named_modules():
    if not (hasattr(mod, "lora_A") and "default" in mod.lora_A): continue
    name = n.replace("base_model.model.", "").replace(".base_layer", "")
    if name not in PIECES: continue
    dt = mod.lora_A["default"].weight.dtype
    mod.lora_A["default"].weight.data = PIECES[name][2].to("cuda", dt)
    MODS[name] = (mod, PIECES[name][0].to("cuda"), PIECES[name][1].to("cuda"), dt)
assert len(MODS) == len(PIECES), f"{len(MODS)} of {len(PIECES)}"
print(f"{len(MODS)} matrices mounted in one adapter of rank {R2}")

@torch.no_grad()
def set_lambdas(lam_L, lam_G):
    for _, (mod, BLc, BGc, dt) in MODS.items():
        mod.lora_B["default"].weight.data = torch.cat([BLc*lam_L, BGc*lam_G], 1).to(dt)

@torch.no_grad()
def bpb_on(X, nbytes, micro=4):
    pm.eval(); nats, i = 0.0, 0
    while i < X.shape[0]:
        try:
            x = X[i:i+micro].to("cuda").long()
            lg = pm(input_ids=x).logits[:, :-1].float(); y = x[:, 1:]
            nats += float(torch.nn.functional.cross_entropy(
                lg.reshape(-1, lg.shape[-1]), y.reshape(-1), reduction="sum"))
            del lg, x, y; i += micro
        except torch.cuda.OutOfMemoryError:
            free(); micro = max(1, micro//2)
            if micro == 1: raise
    return nats/math.log(2)/nbytes

168 matrices mounted in one adapter of rank 32


In [22]:
# ── CORE ───────────────────────────────────────────────────────────────
t0 = time.time()
G = np.zeros((len(LAMS), len(LAMS)))
for i, lL in enumerate(LAMS):
    for j, lG in enumerate(LAMS):
        set_lambdas(lL, lG); G[i, j] = bpb_on(X_SEL, B_SEL)
    print(f"  λ_L={lL:.1f} … {(time.time()-t0)/60:.1f} min", end="\r")
print(f"\ngrid finished in {(time.time()-t0)/60:.1f} min")

print(f"\nbpb on the full TRAIN split · rows λ_L, columns λ_G")
print("      " + "".join(f"{l:>9.1f}" for l in LAMS))
for i, lL in enumerate(LAMS):
    print(f"{lL:>5.1f} " + "".join(f"{G[i,j]:>9.4f}" for j in range(len(LAMS))))

iL, iG = np.unravel_index(G.argmin(), G.shape)
LAM_L, LAM_G = LAMS[iL], LAMS[iG]
print(f"\ncoarse optimum on TRAIN: λ_L = {LAM_L}, λ_G = {LAM_G}  ({G[iL,iG]:.4f})")
print(f"  backbone     (0,0)  {G[0,0]:.4f}")
print(f"  language only(1,0)  {G[-1,0]:.4f}")
print(f"  genre only   (0,1)  {G[0,-1]:.4f}")
print(f"  sum          (1,1)  {G[-1,-1]:.4f}")

FINE = {}
if REFINE:
    cand_L = sorted({round(max(0.0, min(1.0, LAM_L+d)), 3) for d in (-0.1, 0, 0.1)})
    cand_G = sorted({round(max(0.0, min(1.0, LAM_G+d)), 3) for d in (-0.1, 0, 0.1)})
    print(f"\nrefining to 0.1 over λ_L ∈ {cand_L} × λ_G ∈ {cand_G}")
    for a in cand_L:
        for b in cand_G:
            set_lambdas(a, b); FINE[(a, b)] = bpb_on(X_SEL, B_SEL)
            print(f"  ({a}, {b}) → {FINE[(a,b)]:.4f}")
    (LAM_L, LAM_G), best = min(FINE.items(), key=lambda kv: kv[1])
    print(f"refined optimum on TRAIN: λ_L = {LAM_L}, λ_G = {LAM_G}  ({best:.4f})")

print(f"\n{'system':<28}{'bpb val':>10}{'bpb TEST':>11}{'Δ TEST vs θ₀':>15}")
print("-" * 64)
tb = RES["θ₀   backbone"]["target·TEST"][0]
POINTS = [("θ₀            (0, 0)", 0.0, 0.0), ("language only (1, 0)", 1.0, 0.0),
          ("genre only    (0, 1)", 0.0, 1.0), ("sum           (1, 1)", 1.0, 1.0),
          ("mean      (0.5, 0.5)", 0.5, 0.5),
          (f"CHOSEN  ({LAM_L}, {LAM_G})", LAM_L, LAM_G)]
FIT = {}
for name, lL, lG in POINTS:
    set_lambdas(lL, lG)
    v = bpb_on(EVAL["target·val"],  BYTES["target·val"])
    t = bpb_on(EVAL["target·TEST"], BYTES["target·TEST"])
    FIT[name] = (v, t)
    print(f"{name:<28}{v:>10.4f}{t:>11.4f}{t-tb:>+15.4f}")

print("\nTEST surface, inspected AFTER the choice was made, to see what selecting")
print("on train costs; with two parameters it should cost very little.")
GT = np.zeros_like(G)
for i, lL in enumerate(LAMS):
    for j, lG in enumerate(LAMS):
        set_lambdas(lL, lG); GT[i, j] = bpb_on(EVAL["target·TEST"], BYTES["target·TEST"])
print("      " + "".join(f"{l:>9.1f}" for l in LAMS))
for i, lL in enumerate(LAMS):
    print(f"{lL:>5.1f} " + "".join(f"{GT[i,j]:>9.4f}" for j in range(len(LAMS))))
jL, jG = np.unravel_index(GT.argmin(), GT.shape)
chosen_test = FIT[f"CHOSEN  ({LAM_L}, {LAM_G})"][1]
print(f"  true (coarse) optimum on TEST: λ_L = {LAMS[jL]}, λ_G = {LAMS[jG]} ({GT[jL,jG]:.4f})")
print(f"  the train-selected point scores {chosen_test:.4f} on TEST "
      f"→ {chosen_test-GT[jL,jG]:+.4f} bpb for selecting on train")

print(f"\nthe DIAGONAL (λ_L = λ_G), i.e. the scale curve:")
print(f"{'λ':>6}{'train':>10}{'TEST':>10}")
for k, l in enumerate(LAMS):
    print(f"{l:>6.1f}{G[k,k]:>10.4f}{GT[k,k]:>10.4f}")

try: base = pm.unload()
except AttributeError: base = pm.base_model.unload()
del pm, MODS; free()
check_base("lambda grid")

json.dump({"step": STEP_L, "lambdas": LAMS, "train": G.tolist(), "test": GT.tolist(),
           "refined": {f"{a},{b}": v for (a, b), v in FINE.items()},
           "chosen": [LAM_L, LAM_G], "coarse_test_optimum": [LAMS[jL], LAMS[jG]],
           "points": {k: list(v) for k, v in FIT.items()},
           "n_fitted_parameters": 2, "train_blocks": int(X_SEL.shape[0])},
          open(f"{CKPT}/lambdas.json", "w"), indent=1)
print(f"\n→ {CKPT}/lambdas.json")

  λ_L=1.0 … 5.4 min
grid finished in 5.4 min

bpb on the full TRAIN split · rows λ_L, columns λ_G
            0.0      0.2      0.4      0.6      0.8      1.0
  0.0    1.1369   1.1263   1.1249   1.1307   1.1432   1.1625
  0.2    1.1316   1.1224   1.1217   1.1278   1.1402   1.1591
  0.4    1.1319   1.1234   1.1228   1.1288   1.1409   1.1593
  0.6    1.1375   1.1288   1.1277   1.1332   1.1448   1.1628
  0.8    1.1473   1.1380   1.1361   1.1408   1.1518   1.1693
  1.0    1.1603   1.1504   1.1475   1.1514   1.1616   1.1786

coarse optimum on TRAIN: λ_L = 0.2, λ_G = 0.4  (1.1217)
  backbone     (0,0)  1.1369
  language only(1,0)  1.1603
  genre only   (0,1)  1.1625
  sum          (1,1)  1.1786

refining to 0.1 over λ_L ∈ [0.1, 0.2, 0.3] × λ_G ∈ [0.3, 0.4, 0.5]
  (0.1, 0.3) → 1.1224
  (0.1, 0.4) → 1.1228
  (0.1, 0.5) → 1.1250
  (0.2, 0.3) → 1.1212
  (0.2, 0.4) → 1.1217
  (0.2, 0.5) → 1.1240
  (0.3, 0.3) → 1.1211
  (0.3, 0.4) → 1.1218
  (0.3, 0.5) → 1.1240
refined optimum on TRAIN: λ_L = 0.3,

## 12 · One λ per layer

48 scalars instead of 2, initialised at the global optimum so that the two fits
are nested and any gain is attributable to depth alone. The λ are left
**unconstrained**: since τ_L hurts the target on its own, a layer that wants
λ_L < 0 would be a result, and bounding them to [0,1] would hide it.

Two implementation notes. The mixture wrapper works in bf16 and never
materialises fp32 activations — doing so for 168 wrapped matrices costs tens of
gigabytes. Gradient checkpointing is enabled with `use_reentrant=False`: with
`True` nothing would propagate, because the backbone is frozen and the only
parameters are the λ. The whole block runs inside `try/finally`, so the wrappers
come off even if it fails.

In [23]:
# ── CORE ───────────────────────────────────────────────────────────────
import torch.nn as nn, torch.nn.functional as F

LR_LAM     = 5e-3
LAM_STEPS  = 150
EPOCH_CAP  = 3
MICRO_LAM  = 2

assert not [n for n, m in base.named_modules() if type(m).__name__ == "Mix"], \
    "Mix wrappers are still installed; restart from a clean backbone"
free()

shared  = sorted(set(TAU_L) & set(TAU_G))
layers  = sorted({_layer_of(n) for n in shared})
KEYS    = layers
key_of  = lambda n: _layer_of(n)
IDXK    = {k: i for i, k in enumerate(KEYS)}
print(f"{len(KEYS)} positions × 2 modules = {2*len(KEYS)} trainable scalars")

try:
    INI_L, INI_G = float(LAM_L), float(LAM_G)
    print(f"initialised at the grid's global optimum: ({INI_L}, {INI_G})")
except NameError:
    INI_L, INI_G = 0.5, 0.5
    print("grid optimum not found → initialising at (0.5, 0.5)")

class Lambdas(nn.Module):
    def __init__(self, n, iL, iG):
        super().__init__()
        self.L = nn.Parameter(torch.full((n,), iL, dtype=torch.float32))
        self.G = nn.Parameter(torch.full((n,), iG, dtype=torch.float32))
LAM = Lambdas(len(KEYS), INI_L, INI_G).to("cuda")

class Mix(nn.Module):
    """y = Wx + λ_L·s_L·B_L(A_L x) + λ_G·s_G·B_G(A_G x).

    W, A and B are frozen; the ONLY parameters are the λ, shared by every matrix
    at the same position. Everything runs in the backbone dtype: λ is cast to
    bf16 to multiply and its gradient returns to fp32 through the cast."""
    def __init__(self, lin, tl, tg, lam, i):
        super().__init__()
        self.base = lin
        BL, AL, sL = tl; BG, AG, sG = tg
        dt = lin.weight.dtype
        self.register_buffer("AL", (AL*sL).to("cuda", dt))
        self.register_buffer("BL", BL.to("cuda", dt))
        self.register_buffer("AG", (AG*sG).to("cuda", dt))
        self.register_buffer("BG", BG.to("cuda", dt))
        self._lam = [lam]      # inside a list so it is not registered as a submodule
        self.i = i
    def forward(self, x):
        y   = self.base(x)
        lam = self._lam[0]; dt = y.dtype
        dL = F.linear(F.linear(x, self.AL), self.BL)
        dG = F.linear(F.linear(x, self.AG), self.BG)
        return y + lam.L[self.i].to(dt)*dL + lam.G[self.i].to(dt)*dG

def parent(root, path):
    obj = root
    for p in path.split(".")[:-1]: obj = getattr(obj, p)
    return obj, path.split(".")[-1]

ORIG = {}
def unwrap():
    """Tolerant to a partial ORIG: undoes whatever got wrapped."""
    for n in list(ORIG):
        p, a = parent(base, n); setattr(p, a, ORIG[n])
    ORIG.clear()

def fwd_bwd_lam(m, xb, micro):
    """Like fwd_bwd, but it DOES try micro = 1 before giving up."""
    while True:
        try:
            total = 0.0
            for i in range(0, xb.shape[0], micro):
                x = xb[i:i+micro].to("cuda", non_blocking=True).long()
                out = m(input_ids=x, labels=x)
                (out.loss * x.shape[0] / xb.shape[0]).backward()
                total += float(out.loss.detach()) * x.shape[0]
                del out, x
            return total/xb.shape[0], micro
        except torch.cuda.OutOfMemoryError:
            m.zero_grad(set_to_none=True); free()
            if micro == 1:
                raise RuntimeError("OOM at micro=1: not even one block fits.")
            micro = max(1, micro//2)
            print(f"      OOM → micro={micro}")

ckpt_on = False
try:
    for n in shared:
        p, a = parent(base, n)
        ORIG[n] = getattr(p, a)
        setattr(p, a, Mix(ORIG[n], TAU_L[n], TAU_G[n], LAM, IDXK[key_of(n)]))
    for q in base.parameters(): q.requires_grad_(False)
    for q in LAM.parameters():  q.requires_grad_(True)
    n_par = sum(q.numel() for q in LAM.parameters() if q.requires_grad)
    print(f"{len(shared)} matrices wrapped · {n_par} trainable parameters")

    # With λ constant at the global optimum this MUST reproduce the grid. It is
    # also the check that doing the product in bf16 changed nothing.
    v0 = measure(base, "target·val")[0]
    ref = FIT[f"CHOSEN  ({LAM_L}, {LAM_G})"][0]
    print(f"\ncheck · with λ constant at ({INI_L}, {INI_G}) validation gives {v0:.4f}, "
          f"the grid gave {ref:.4f} (difference {v0-ref:+.4f})")
    assert abs(v0-ref) < 5e-3, "does not match the grid: something is mounted wrongly"

    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.config.use_cache = False
    ckpt_on = True
    print("gradient checkpointing enabled (use_reentrant=False)")

    X_TR  = flat_blocks(TGT["train"], "target·train")
    steps = int(min(LAM_STEPS, max(30, EPOCH_CAP*X_TR.shape[0]/BATCH)))
    opt   = torch.optim.AdamW(LAM.parameters(), lr=LR_LAM, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    g     = torch.Generator().manual_seed(SEED)
    order = torch.randperm(X_TR.shape[0], generator=g)
    micro, t0 = MICRO_LAM, time.time()
    print(f"\n{steps} steps over {X_TR.shape[0]} training blocks "
          f"({EPOCH_CAP} epochs at most) · initial micro {micro}")
    for step in range(1, steps+1):
        start = ((step-1)*BATCH) % X_TR.shape[0]
        idx = order[start:start+BATCH]
        if idx.shape[0] < BATCH: idx = torch.cat([idx, order[:BATCH-idx.shape[0]]])
        opt.zero_grad(set_to_none=True)
        loss, micro = fwd_bwd_lam(base, X_TR[idx], micro)
        opt.step(); sched.step()
        if step % 25 == 0 or step == steps:
            print(f"  step {step:>4}/{steps} · loss {loss:.4f} · "
                  f"mean λ_L {float(LAM.L.mean()):+.3f} · mean λ_G {float(LAM.G.mean()):+.3f}"
                  f" · peak GPU {torch.cuda.max_memory_allocated()/2**30:.1f} GiB"
                  f" · {time.time()-t0:.0f}s")

    base.gradient_checkpointing_disable(); ckpt_on = False
    lamL = LAM.L.detach().float().cpu().numpy()
    lamG = LAM.G.detach().float().cpu().numpy()
    vv = measure(base, "target·val"); tt = measure(base, "target·TEST")
    tb = RES["θ₀   backbone"]["target·TEST"][0]
    print(f"\n{'system':<26}{'bpb val':>10}{'bpb TEST':>11}{'Δ TEST vs θ₀':>15}")
    print("-" * 62)
    print(f"{'θ₀   backbone':<26}{RES['θ₀   backbone']['target·val'][0]:>10.4f}"
          f"{tb:>11.4f}{0.0:>+15.4f}")
    gname = f"CHOSEN  ({LAM_L}, {LAM_G})"
    print(f"{'global (2 scalars)':<26}{FIT[gname][0]:>10.4f}{FIT[gname][1]:>11.4f}"
          f"{FIT[gname][1]-tb:>+15.4f}")
    print(f"{f'per layer ({n_par} scalars)':<26}{vv[0]:>10.4f}{tt[0]:>11.4f}{tt[0]-tb:>+15.4f}")
    print(f"\nwhat depth buys over the global fit: {tt[0]-FIT[gname][1]:+.4f} bpb")
    print("(nested test: the per-layer fit starts AT the global one, so it can only")
    print(" improve it on train; if it does not improve on test, 2 scalars sufficed)")

    print(f"\nTHE DEPTH PROFILE")
    print(f"{'layer':>6}{'λ_L':>9}{'λ_G':>9}   bar (█ = 0.05)")
    for i, c in enumerate(KEYS):
        bar = lambda v: ("−" if v < 0 else "") + "█"*int(round(abs(v)/0.05))
        print(f"{c:>6}{lamL[i]:>+9.3f}{lamG[i]:>+9.3f}   "
              f"L:{bar(lamL[i]):<14} G:{bar(lamG[i])}")
    ends = [0, 1, len(KEYS)-2, len(KEYS)-1]
    inner = [i for i in range(len(KEYS)) if i not in ends]
    print(f"\n  terminal blocks: λ_L {lamL[ends].mean():+.3f} · λ_G {lamG[ends].mean():+.3f}")
    print(f"  interior       : λ_L {lamL[inner].mean():+.3f} · λ_G {lamG[inner].mean():+.3f}")
    print(f"  correlation between the two profiles across depth: "
          f"{float(np.corrcoef(lamL, lamG)[0,1]):+.3f}")
    print(f"  layers with λ_L < 0: {int((lamL<0).sum())} of {len(KEYS)} · "
          f"with λ_G < 0: {int((lamG<0).sum())} of {len(KEYS)}")
    print(f"\ngeneralisation gap: fitted on TRAIN, val {vv[0]:.4f} · test {tt[0]:.4f} "
          f"({tt[0]-vv[0]:+.4f})")
    print("NOTE: 48 free parameters, one seed, no interval. Against the 2 scalars,")
    print("which do have an overfitting diagnostic, this fit is exploratory.")

    json.dump({"n_parameters": int(n_par), "init": [INI_L, INI_G], "steps": steps,
               "lr": LR_LAM, "lam_L": lamL.tolist(), "lam_G": lamG.tolist(),
               "val": list(vv), "test": list(tt),
               "gain_over_global": float(tt[0]-FIT[gname][1])},
              open(f"{CKPT}/lambdas_per_layer.json", "w"), indent=1)
    print(f"\n→ {CKPT}/lambdas_per_layer.json")

finally:
    if ckpt_on:
        try: base.gradient_checkpointing_disable()
        except Exception: pass
    unwrap()
    try: del LAM
    except NameError: pass
    free()
    left = sum(1 for _, m in base.named_modules() if type(m).__name__ == "Mix")
    assert left == 0, f"{left} wrappers left"
    check_base("per-layer lambdas")
    print("backbone restored and verified ✓")

24 positions × 2 modules = 48 trainable scalars
initialised at the grid's global optimum: (0.3, 0.3)
168 matrices wrapped · 48 trainable parameters

check · with λ constant at (0.3, 0.3) validation gives 1.1215, the grid gave 1.1215 (difference +0.0000)
gradient checkpointing enabled (use_reentrant=False)
target·train: 366 blocks · 187,392 tokens

68 steps over 366 training blocks (3 epochs at most) · initial micro 2


/tmp/ipykernel_3474/1764874687.py:131: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  f"mean λ_L {float(LAM.L.mean()):+.3f} · mean λ_G {float(LAM.G.mean()):+.3f}"


  step   25/68 · loss 3.0594 · mean λ_L +0.255 · mean λ_G +0.310 · peak GPU 8.6 GiB · 78s
  step   50/68 · loss 3.1247 · mean λ_L +0.242 · mean λ_G +0.314 · peak GPU 8.6 GiB · 156s
  step   68/68 · loss 3.1750 · mean λ_L +0.241 · mean λ_G +0.315 · peak GPU 8.6 GiB · 213s

system                       bpb val   bpb TEST   Δ TEST vs θ₀
--------------------------------------------------------------
θ₀   backbone                 1.1396     1.1045        +0.0000
global (2 scalars)            1.1215     1.0888        -0.0157
per layer (48 scalars)        1.1188     1.0869        -0.0176

what depth buys over the global fit: -0.0019 bpb
(nested test: the per-layer fit starts AT the global one, so it can only
 improve it on train; if it does not improve on test, 2 scalars sufficed)

THE DEPTH PROFILE
 layer      λ_L      λ_G   bar (█ = 0.05)
     0   +0.427   +0.199   L:█████████      G:████
     1   +0.355   +0.286   L:███████        G:██████
     2   +0.222   +0.241   L:████           G:████

## 13 · The control that decides the paper

The composition has to be compared against **a single module with its own fitted
scale**, not against an unscaled one. Otherwise the comparison confounds *adding
the second factor* with *turning the first one down*, and separating those two is
the whole point of the study.

In [24]:
# ── CORE ───────────────────────────────────────────────────────────────
LL, LG   = float(LAM_L), float(LAM_G)
LG_ALONE = LAMS[int(np.argmin(G[0]))]          # best λ_G with λ_L = 0, chosen on TRAIN
LL_ALONE = LAMS[int(np.argmin(G[:, 0]))]       # best λ_L with λ_G = 0, chosen on TRAIN
print(f"λ_G with λ_L=0 selected on train: {LG_ALONE} (train {G[0].min():.4f})")
print(f"λ_L with λ_G=0 selected on train: {LL_ALONE} (train {G[:,0].min():.4f})")

NEW = [(f"genre only    (0,{LG_ALONE})", [(TAU_G, LG_ALONE, None)]),
       (f"language only ({LL_ALONE},0)", [(TAU_L, LL_ALONE, None)]),
       (f"composition   ({LL},{LG})",    [(TAU_L, LL, None), (TAU_G, LG, None)])]
for label, recipe in NEW:
    NATS[label] = nats_of_system(combine(recipe))
    print(f"  {label:<28} bpb {NATS[label].sum()/math.log(2)/BYTES_BLK.sum():.4f}")

comp, genre_only, lang_only = NEW[2][0], NEW[0][0], NEW[1][0]
m, lo, hi = ci(NATS[comp], NATS[genre_only], "chapter")
print(f"\nMARGINAL CONTRIBUTION OF THE LANGUAGE FACTOR")
print(f"  {comp} − {genre_only}")
print(f"  Δ = {m:+.4f} bpb · CI95 by chapter [{lo:+.4f}, {hi:+.4f}]"
      f"{'   ✱ excludes zero' if not (lo <= 0 <= hi) else '   — includes zero'}")
wins = 0
for c in chapters:
    s = CHAPLAB == c
    d = (NATS[comp][s].sum()-NATS[genre_only][s].sum())/math.log(2)/BYTES_BLK[s].sum()
    wins += d < 0
    print(f"    chapter {c}: {d:+.4f}")
print(f"  in favour of the composition: {wins}/{len(chapters)}")

mg, log_, hig = ci(NATS[genre_only], NATS["θ₀   backbone"], "chapter")
ml, lol, hil  = ci(NATS[lang_only],  NATS["θ₀   backbone"], "chapter")
mt, lot, hit  = ci(NATS[comp],       NATS["θ₀   backbone"], "chapter")
print(f"\n  genre only, scaled    − θ₀ = {mg:+.4f}  CI95 [{log_:+.4f}, {hig:+.4f}]")
print(f"  language only, scaled − θ₀ = {ml:+.4f}  CI95 [{lol:+.4f}, {hil:+.4f}]")
print(f"  composition           − θ₀ = {mt:+.4f}  CI95 [{lot:+.4f}, {hit:+.4f}]")
print(f"\nAttribution depends on the order of the decomposition, so report both:")
print(f"  genre first    → language is worth {100*abs(m)/abs(mt):.0f} % of the total")
print(f"  language first → genre    is worth {100*abs(mt-ml)/abs(mt):.0f} % of the total")
sl = (abs(ml) + abs(m))/2; sg = (abs(mg) + abs(mt-ml))/2
print(f"  averaging the two orders → language {100*sl/(sl+sg):.0f} %, genre {100*sg/(sl+sg):.0f} %")
print(f"  the two scaled factors are sub-additive: {abs(mg)+abs(ml):.4f} separately "
      f"against {abs(mt):.4f} together ({100*(abs(mg)+abs(ml)-abs(mt))/(abs(mg)+abs(ml)):.1f} % redundant)")

json.dump({"marginal_language": [m, lo, hi], "genre_scaled_vs_backbone": [mg, log_, hig],
           "language_scaled_vs_backbone": [ml, lol, hil], "composition_vs_backbone": [mt, lot, hit],
           "lambda_G_alone": LG_ALONE, "lambda_L_alone": LL_ALONE, "lambda_comp": [LL, LG]},
          open(f"{CKPT}/single_module_control.json", "w"), indent=1)
print(f"\n→ {CKPT}/single_module_control.json")

λ_G with λ_L=0 selected on train: 0.4 (train 1.1249)
λ_L with λ_G=0 selected on train: 0.2 (train 1.1316)
  genre only    (0,0.4)        bpb 1.0934
  language only (0.2,0)        bpb 1.0997
  composition   (0.3,0.3)      bpb 1.0893

MARGINAL CONTRIBUTION OF THE LANGUAGE FACTOR
  composition   (0.3,0.3) − genre only    (0,0.4)
  Δ = -0.0041 bpb · CI95 by chapter [-0.0045, -0.0036]   ✱ excludes zero
    chapter 15: -0.0045
    chapter 16: -0.0052
    chapter 17: -0.0037
    chapter 18: -0.0039
    chapter 19: -0.0034
  in favour of the composition: 5/5

  genre only, scaled    − θ₀ = -0.0118  CI95 [-0.0155, -0.0091]
  language only, scaled − θ₀ = -0.0053  CI95 [-0.0057, -0.0051]
  composition           − θ₀ = -0.0159  CI95 [-0.0190, -0.0135]

Attribution depends on the order of the decomposition, so report both:
  genre first    → language is worth 26 % of the total
  language first → genre    is worth 67 % of the total
  averaging the two orders → language 30 %, genre 70 %
  the two sca

## 14 · Where the gains and the losses come from

Per-token NLL for every system on the test split, decomposed by position in the
block, by which module saw each token during its own training, by corpus
frequency, and by token class. The "which module saw it" split is **exact**, not
estimated: the training blocks are still in memory and can simply be counted.

One caveat that belongs with the table. Bytes are attributed token by token, and
decoding tokens in isolation loses leading spaces, so about 16 % of the true byte
count goes missing. That loss falls almost entirely on word-initial tokens, so
comparisons *down* a column are sound while comparisons *across* groups carry a
bias we have not bounded.

In [25]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
X   = EVAL["target·TEST"]
IDS = X[:, 1:].reshape(-1).long().numpy()
POS = np.tile(np.arange(X.shape[1]-1), X.shape[0])
print(f"test: {X.shape[0]} blocks × {X.shape[1]-1} predicted positions "
      f"= {len(IDS):,} tokens")

@torch.no_grad()
def nll_per_token(adapter, micro=4):
    global base
    m, pm = base, None
    if adapter is not None:
        R2 = max(B.shape[1] for B, _ in adapter.values())
        cfg = LoraConfig(r=R2, lora_alpha=R2, lora_dropout=0.0, bias="none",
                         target_modules=TARGETS, task_type="CAUSAL_LM")
        pm = get_peft_model(base, cfg)
        for n, mod in pm.named_modules():
            if not (hasattr(mod, "lora_A") and "default" in mod.lora_A): continue
            name = n.replace("base_model.model.", "").replace(".base_layer", "")
            if name not in adapter: continue
            B_, A_ = adapter[name]; dt = mod.lora_A["default"].weight.dtype
            mod.lora_A["default"].weight.data = A_.to("cuda", dt)
            mod.lora_B["default"].weight.data = B_.to("cuda", dt)
        m = pm
    m.eval(); out, i = [], 0
    while i < X.shape[0]:
        try:
            x = X[i:i+micro].to("cuda").long()
            lg = m(input_ids=x).logits[:, :-1].float(); y = x[:, 1:]
            l = torch.nn.functional.cross_entropy(
                lg.reshape(-1, lg.shape[-1]), y.reshape(-1), reduction="none")
            out.append(l.view(y.shape).cpu().numpy()); del lg, x, y, l; i += micro
        except torch.cuda.OutOfMemoryError:
            free(); micro = max(1, micro//2)
            if micro == 1: raise
    if pm is not None:
        try: base = pm.unload()
        except AttributeError: base = pm.base_model.unload()
        del pm; free()
    return np.concatenate(out, 0).reshape(-1)

test: 129 blocks × 511 predicted positions = 65,919 tokens


In [26]:
# ── CORE ───────────────────────────────────────────────────────────────
COMP = f"COMP({LL},{LG})"
RECIPES = {"θ₀":   None,
           "τ_L":  [(TAU_L, 1.0, None)],
           "τ_G":  [(TAU_G, 1.0, None)],
           "mean": [(TAU_L, 0.5, None), (TAU_G, 0.5, None)],
           COMP:   [(TAU_L, LL, None), (TAU_G, LG, None)]}
NLL = {}
for name, recipe in RECIPES.items():
    NLL[name] = nll_per_token(None if recipe is None else combine(recipe))
    print(f"  {name:<14} mean nll {NLL[name].mean():.4f}")

uni   = np.unique(IDS)
BY_ID = np.zeros(int(uni.max())+1, dtype=np.int64)
for t in uni: BY_ID[t] = max(1, len(tok.decode([int(t)]).encode("utf-8")))
BYT = BY_ID[IDS]
print(f"\nbytes attributed token by token: {BYT.sum():,} · true byte count: "
      f"{BYTES['target·TEST']:,} "
      f"({100*(BYT.sum()-BYTES['target·TEST'])/BYTES['target·TEST']:+.2f} %)")

def dbpb(a, b, mask):
    """Δbpb between systems a and b, restricted to `mask`."""
    if mask.sum() == 0: return float("nan")
    return (NLL[a][mask].sum()-NLL[b][mask].sum())/math.log(2)/BYT[mask].sum()

COLS = (("COMP", COMP, "θ₀"), ("mean", "mean", "θ₀"),
        ("τ_L", "τ_L", "θ₀"), ("τ_G", "τ_G", "θ₀"), ("COMP−τ_G", COMP, "τ_G"))
def table(masks, title):
    print(f"\n{title}")
    print(f"{'group':<26}{'tokens':>9}{'% bytes':>9}" + "".join(f"{c[0]:>11}" for c in COLS))
    print("-"*(44+11*len(COLS)))
    for name, m in masks:
        row = f"{name:<26}{int(m.sum()):>9,}{100*BYT[m].sum()/BYT.sum():>8.1f}%"
        for _, a, b in COLS: row += f"{dbpb(a,b,m):>+11.4f}"
        print(row)

  θ₀             mean nll 3.1238
  τ_L            mean nll 3.1907
  τ_G            mean nll 3.1900
  mean           mean nll 3.0945
  COMP(0.3,0.3)  mean nll 3.0794

bytes attributed token by token: 225,904 · true byte count: 268,963 (-16.01 %)


In [27]:
# ── CORE ───────────────────────────────────────────────────────────────
NB = 8; width = (X.shape[1]-1)//NB
table([(f"positions {k*width}–{(k+1)*width-1}",
        (POS >= k*width) & (POS < (k+1)*width)) for k in range(NB)],
      "A · BY POSITION WITHIN THE BLOCK (Δbpb; negative = better than θ₀)")
print("If the gain concentrates at the start of the block, the composition is")
print("supplying a prior where there is little context; if it is flat, it is a")
print("general displacement of the model rather than a context effect.")

V = int(BY_ID.shape[0])
def count_tokens(T, V, chunk=4_000_000):
    c = np.zeros(V, dtype=np.int64); f = T.reshape(-1)
    for i in range(0, f.shape[0], chunk):
        c += torch.bincount(f[i:i+chunk].long(), minlength=V).numpy()[:V]
    return c
CNT_L = count_tokens(CORPUS["language"], V)
CNT_G = count_tokens(CORPUS["genre"], V)
print(f"\nexact counts: language corpus {CORPUS['language'].numel():,} tokens · "
      f"genre corpus {CORPUS['genre'].numel():,}")

nL, nG = CNT_L[IDS], CNT_G[IDS]
table([("neither module saw it", (nL == 0) & (nG == 0)),
       ("only the language one", (nL >  0) & (nG == 0)),
       ("only the genre one",    (nL == 0) & (nG >  0)),
       ("both",                  (nL >  0) & (nG >  0))],
      "B1 · BY WHICH MODULE SAW THE TOKEN DURING ITS OWN TRAINING")

BINS = [(0,0),(1,10),(11,100),(101,1000),(1001,10**9)]
table([(f"genre corpus: {a}–{b if b<10**8 else '∞'}", (nG>=a)&(nG<=b)) for a,b in BINS],
      "B2 · BY TOKEN FREQUENCY IN THE GENRE CORPUS")
table([(f"language corpus: {a}–{b if b<10**8 else '∞'}", (nL>=a)&(nL<=b)) for a,b in BINS],
      "B3 · BY TOKEN FREQUENCY IN THE LANGUAGE CORPUS")

TXT = {int(t): tok.convert_ids_to_tokens([int(t)])[0] for t in uni}
def token_class(t):
    s = TXT.get(int(t), ""); n = s.lstrip("▁")
    if s.startswith("▁") and n and n[0].isalpha(): return "word-initial"
    if n and n[0].isalpha():                       return "continuation"
    if any(c.isdigit() for c in n):                return "digits"
    if n.strip() == "":                            return "whitespace"
    return "punctuation and other"
CLS = np.array([token_class(t) for t in IDS])
table([(c, CLS == c) for c in ("word-initial", "continuation",
                               "punctuation and other", "digits", "whitespace")],
      "C · BY TOKEN CLASS")

def by_token(a, b, top=12):
    d = (NLL[a]-NLL[b])/math.log(2)
    agg, cnt = Counter(), Counter()
    for t, v in zip(IDS, d):
        agg[int(t)] += float(v); cnt[int(t)] += 1
    rows = [(t, agg[t], cnt[t]) for t in agg if cnt[t] >= 20]
    rows.sort(key=lambda r: r[1])
    return rows[:top], rows[-top:]

best_t, worst_t = by_token(COMP, "θ₀")
print(f"\nD · TOKENS CONTRIBUTING MOST TO {COMP} − θ₀ (≥20 occurrences)")
print(f"{'largest gains':<40}{'largest losses'}")
print(f"{'token':>14}{'Δbits':>9}{'n':>6}   |{'token':>14}{'Δbits':>9}{'n':>6}")
for (t1,s1,n1), (t2,s2,n2) in zip(best_t, reversed(worst_t)):
    print(f"{TXT.get(t1,'?')[:13]:>14}{s1:>9.1f}{n1:>6}   |"
          f"{TXT.get(t2,'?')[:13]:>14}{s2:>9.1f}{n2:>6}")
tot_bits = (NLL["θ₀"].sum()-NLL[COMP].sum())/math.log(2)
print(f"  the single largest gainer is {abs(best_t[0][1])/tot_bits*100:.0f} % of the "
      f"{tot_bits:.0f} bits the composition gains in total")

_, worstL = by_token("τ_L", "θ₀")
print(f"\nE · TOKENS MOST DAMAGED BY τ_L RELATIVE TO θ₀ (≥20 occurrences)")
print(f"{'token':>14}{'Δbits':>9}{'n':>6}   class")
for t, s, n in reversed(worstL):
    print(f"{TXT.get(t,'?')[:13]:>14}{s:>9.1f}{n:>6}   {token_class(t)}")

json.dump({"systems": {k: float(v.mean()) for k, v in NLL.items()},
           "bytes_attributed": int(BYT.sum()), "bytes_true": int(BYTES["target·TEST"]),
           "by_position": {f"{k*width}-{(k+1)*width-1}":
                           dbpb(COMP, "θ₀", (POS>=k*width)&(POS<(k+1)*width))
                           for k in range(NB)},
           "by_class": {c: dbpb(COMP, "θ₀", CLS == c) for c in set(CLS.tolist())},
           "by_seen": {"neither": dbpb(COMP,"θ₀",(nL==0)&(nG==0)),
                       "language_only": dbpb(COMP,"θ₀",(nL>0)&(nG==0)),
                       "genre_only": dbpb(COMP,"θ₀",(nL==0)&(nG>0)),
                       "both": dbpb(COMP,"θ₀",(nL>0)&(nG>0))}},
          open(f"{CKPT}/gain_profile.json","w"), indent=1, ensure_ascii=False)
print(f"\n→ {CKPT}/gain_profile.json")
check_base("gain profile")


A · BY POSITION WITHIN THE BLOCK (Δbpb; negative = better than θ₀)
group                        tokens  % bytes       COMP       mean        τ_L        τ_G   COMP−τ_G
---------------------------------------------------------------------------------------------------
positions 0–62                8,127    12.3%    -0.0675    -0.0670    +0.0576    +0.0102    -0.0778
positions 63–125              8,127    12.3%    -0.0255    -0.0176    +0.0293    +0.0202    -0.0457
positions 126–188             8,127    12.5%    -0.0174    -0.0120    +0.0223    +0.0217    -0.0390
positions 189–251             8,127    12.3%    -0.0120    -0.0055    +0.0219    +0.0280    -0.0401
positions 252–314             8,127    12.5%    -0.0086    -0.0008    +0.0225    +0.0315    -0.0400
positions 315–377             8,127    12.3%    -0.0077    -0.0009    +0.0227    +0.0346    -0.0424
positions 378–440             8,127    12.1%    -0.0082    -0.0015    +0.0245    +0.0341    -0.0423
positions 441–503             8,

True

## 15 · Do the representations differ, and how?

In weight space the two modules are close to orthogonal, and the plain sum still
overshoots. That leaves one hypothesis alive: the interference exists but in
**representation** space — the two modules push the activations in the same
direction while writing into different weight coordinates.

**The null here is not the one used for principal angles.** There the cosines
were singular values, non-negative by construction, with chance at `√(r/d)`. Here
the cosine carries a **sign** and its null is **zero**; `1/√d` is the typical
scale of a random cosine, not its expectation.

In [28]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
@torch.no_grad()
def hidden(adapter, Xin, stride, micro=2):
    """Hidden states of every layer, subsampled, on CPU fp16.
    Returns a (layers, n_vectors, d) tensor."""
    global base
    m, pm = base, None
    if adapter is not None:
        R2 = max(B.shape[1] for B, _ in adapter.values())
        cfg = LoraConfig(r=R2, lora_alpha=R2, lora_dropout=0.0, bias="none",
                         target_modules=TARGETS, task_type="CAUSAL_LM")
        pm = get_peft_model(base, cfg)
        for n, mod in pm.named_modules():
            if not (hasattr(mod, "lora_A") and "default" in mod.lora_A): continue
            name = n.replace("base_model.model.", "").replace(".base_layer", "")
            if name not in adapter: continue
            B_, A_ = adapter[name]; dt = mod.lora_A["default"].weight.dtype
            mod.lora_A["default"].weight.data = A_.to("cuda", dt)
            mod.lora_B["default"].weight.data = B_.to("cuda", dt)
        m = pm
    m.eval(); chunks = []
    for i in range(0, Xin.shape[0], micro):
        x = Xin[i:i+micro].to("cuda").long()
        hs = m(input_ids=x, output_hidden_states=True).hidden_states
        chunks.append(torch.stack([h[:, ::stride].reshape(-1, h.shape[-1]).half().cpu()
                                   for h in hs], 0))
        del hs, x
    free()
    if pm is not None:
        try: base = pm.unload()
        except AttributeError: base = pm.base_model.unload()
        del pm; free()
    return torch.cat(chunks, 1)

In [29]:
# ── CORE ───────────────────────────────────────────────────────────────
N_BLK_H = 16     # test blocks used for the hidden states
STRIDE  = 4      # one position in every STRIDE

Xh = EVAL["target·TEST"][:N_BLK_H]
print(f"hidden states over {N_BLK_H} blocks × {Xh.shape[1]//STRIDE} positions "
      f"= {N_BLK_H*(Xh.shape[1]//STRIDE):,} vectors per layer")

H0 = hidden(None, Xh, STRIDE)
HL = hidden(combine([(TAU_L, 1.0, None)]), Xh, STRIDE)
HG = hidden(combine([(TAU_G, 1.0, None)]), Xh, STRIDE)
HC = hidden(combine([(TAU_L, LL, None), (TAU_G, LG, None)]), Xh, STRIDE)
dL, dG, dC = (HL-H0).float(), (HG-H0).float(), (HC-H0).float()
H0f = H0.float()
NC, n_vec, d_model = H0.shape[0], H0.shape[1], H0.shape[-1]
scale = 1.0/math.sqrt(d_model)
sem   = 1.0/math.sqrt(d_model*n_vec)

def rel(dd, h): return float((dd.norm(dim=-1)/h.norm(dim=-1).clamp_min(1e-6)).mean())
def cos(a, b):
    num = (a*b).sum(-1); den = a.norm(dim=-1)*b.norm(dim=-1)
    return float((num/den.clamp_min(1e-6)).mean())

print(f"\n{'layer':>6}{'‖δ_L‖/‖h‖':>11}{'‖δ_G‖/‖h‖':>11}{'‖δ_C‖/‖h‖':>11}"
      f"{'cos(δ_L,δ_G)':>14}{'in σ':>8}")
print("-"*62)
COS = []
for c in range(NC):
    cc = cos(dL[c], dG[c]); COS.append(cc)
    print(f"{c:>6}{rel(dL[c],H0f[c]):>11.4f}{rel(dG[c],H0f[c]):>11.4f}"
          f"{rel(dC[c],H0f[c]):>11.4f}{cc:>14.4f}{cc/scale:>7.1f}")
COS = np.array(COS)
print(f"\nNULL: for two independent pushes the SIGNED cosine averages 0.")
print(f"  typical scale of a random cosine in dimension {d_model}: σ = {scale:.4f}")
print(f"  standard error of the mean over {n_vec:,} vectors:        {sem:.5f}")
print(f"mean cosine between the two pushes: {COS[1:].mean():+.4f}")

# the literary ↔ legal axis, built from backbone centroids alone
HJ = hidden(None, EVAL["language·val"][:N_BLK_H], STRIDE)
HT = hidden(None, EVAL["genre·val"][:N_BLK_H], STRIDE)
AXIS = HT.float().mean(1) - HJ.float().mean(1)
AXIS = AXIS / AXIS.norm(dim=-1, keepdim=True).clamp_min(1e-6)
PROJ = {}
for name, dd in (("τ_L", dL), ("τ_G", dG), ("COMP", dC)):
    PROJ[name] = np.array([float((dd[c] @ AXIS[c]).mean() /
                                 dd[c].norm(dim=-1).mean().clamp_min(1e-6))
                           for c in range(NC)])
print(f"\nprojection onto the literary↔legal axis (+ = towards literary)")
for name in PROJ:
    print(f"  {name:>5}: mean {PROJ[name][1:].mean():+.4f} · layers pushing towards "
          f"legal {int((PROJ[name][1:]<0).sum())}/{NC-1}")

c_mean = float(COS[1:].mean())
print(f"\nmean cosine c = {c_mean:.4f}")
print(f"  λ matching the displacement NORM : 1/√(1+c) = {1.0/math.sqrt(1.0+c_mean):.3f}")
print(f"  λ preserving each PROJECTION     : 1/(1+c)  = {1.0/(1.0+c_mean):.3f}")
print(f"  λ actually fitted on the grid    : λ_L={LAM_L}, λ_G={LAM_G}")

json.dump({"cosine_scale": scale, "sem": sem, "cos_per_layer": COS.tolist(),
           "axis_projection": {k: v.tolist() for k, v in PROJ.items()},
           "predicted_lambda": {"norm": 1.0/math.sqrt(1.0+c_mean),
                                "projection": 1.0/(1.0+c_mean)}},
          open(f"{CKPT}/representations.json", "w"), indent=1)
print(f"\n→ {CKPT}/representations.json")
del HL, HG, HC, HJ, HT; free()
check_base("representations")

hidden states over 16 blocks × 128 positions = 2,048 vectors per layer

 layer  ‖δ_L‖/‖h‖  ‖δ_G‖/‖h‖  ‖δ_C‖/‖h‖  cos(δ_L,δ_G)    in σ
--------------------------------------------------------------
     0     0.0000     0.0000     0.0000        0.0000    0.0
     1     0.1037     0.1090     0.0516        0.1963    8.9
     2     0.1419     0.1581     0.0720        0.1321    6.0
     3     0.1621     0.1878     0.0835        0.1325    6.0
     4     0.1863     0.2185     0.0980        0.1415    6.4
     5     0.2155     0.2549     0.1163        0.1151    5.2
     6     0.2259     0.2757     0.1245        0.1197    5.4
     7     0.2308     0.2858     0.1291        0.1328    6.0
     8     0.2360     0.3030     0.1348        0.1571    7.1
     9     0.2423     0.3177     0.1407        0.1472    6.7
    10     0.2463     0.3287     0.1453        0.1570    7.1
    11     0.2469     0.3333     0.1470        0.1577    7.1
    12     0.2467     0.3376     0.1485        0.1630    7.4
    13    

True

## 16 · Two clean axes, and the intervention

The axis above moved both factors at once. Here two axes are built with a single
factor varying at a time — genre as *sv literary − sv legal*, language as
*non-sv literary − sv literary* — so that "τ_L drags the target towards legal"
(its confound) can be told apart from "τ_L pushes it towards Swedish" (its job).

The correction removes from each module its component along the *other* factor's
axis. It can only be applied where the module writes directly into the residual
stream, i.e. `o_proj` and `down_proj`; it folds into the `B` factor, so the
result is still a LoRA adapter of the same rank.

**Protocol.** The axes use the target's validation split, so this probe is *not*
zero-shot: it spends the same supervised budget as the fitted λ and is therefore
compared against it, not against the plain sum. The variant to report is fixed in
advance — both modules cleaned — rather than chosen among the four.

In [30]:
# ── CORE ───────────────────────────────────────────────────────────────
PRESPECIFIED = "+ both cleaned"

CEN = {}
for name, key in (("sv_lit", "target·val"), ("sv_legal", "language·val"),
                  ("nonsv_lit", "genre·val")):
    H = hidden(None, EVAL[key][:N_BLK_H], STRIDE)
    CEN[name] = H.float().mean(1)
    del H; free()

E_GENRE = CEN["sv_lit"]    - CEN["sv_legal"]     # + = more literary
E_LANG  = CEN["nonsv_lit"] - CEN["sv_lit"]       # + = less Swedish
E_GENRE = E_GENRE/E_GENRE.norm(dim=-1, keepdim=True).clamp_min(1e-6)
E_LANG  = E_LANG /E_LANG .norm(dim=-1, keepdim=True).clamp_min(1e-6)
cos_axes = (E_GENRE*E_LANG).sum(-1)
print(f"cosine between the two axes: mean {float(cos_axes[1:].mean()):+.3f} · "
      f"min {float(cos_axes[1:].min()):+.3f} · max {float(cos_axes[1:].max()):+.3f}")
print("(they share the sv-literary centroid with opposite signs, so they are not")
print(" independent; the 2×2 Gram system below handles that correctly)")

def decompose(dd, c):
    """Least-squares fit of dd onto the plane {E_GENRE, E_LANG} in layer c.
    Returns (genre coefficient, language coefficient, fraction of ‖d‖² explained)."""
    u, v = E_GENRE[c], E_LANG[c]
    g = float((u*v).sum())
    du, dv = dd @ u, dd @ v
    det = 1.0 - g*g
    if abs(det) < 1e-6: det = 1e-6
    a, b = (du - g*dv)/det, (dv - g*du)/det
    proj = a[:, None]*u[None, :] + b[:, None]*v[None, :]
    n2 = (dd*dd).sum(-1).clamp_min(1e-9)
    frac = float(((proj*proj).sum(-1)/n2).mean())
    nd = dd.norm(dim=-1).mean().clamp_min(1e-6)
    return float(a.mean())/float(nd), float(b.mean())/float(nd), frac

DEC = {k: {"genre": [], "lang": [], "expl": []} for k in ("τ_L", "τ_G", "COMP")}
for c in range(NC):
    for name, dd in (("τ_L", dL), ("τ_G", dG), ("COMP", dC)):
        a, b, f = decompose(dd[c], c)
        DEC[name]["genre"].append(a); DEC[name]["lang"].append(b); DEC[name]["expl"].append(f)
print(f"\n{'module':>7}{'genre axis':>12}{'language axis':>15}{'plane explains':>16}"
      f"{'max':>8}")
print("-"*58)
for name in DEC:
    g = np.array(DEC[name]["genre"][1:]); l = np.array(DEC[name]["lang"][1:])
    e = np.array(DEC[name]["expl"][1:])
    print(f"{name:>7}{g.mean():>+12.3f}{l.mean():>+15.3f}"
          f"{100*e.mean():>15.1f}%{100*e.max():>7.1f}%")
print("\nExpected if each module did its job: τ_G positive on the genre axis,")
print("τ_L negative on the language axis. Expected if each carries its confound:")
print("τ_L negative on the genre axis, τ_G positive on the language axis.")

cosine between the two axes: mean +0.096 · min -0.532 · max +0.779
(they share the sv-literary centroid with opposite signs, so they are not
 independent; the 2×2 Gram system below handles that correctly)

 module  genre axis  language axis  plane explains     max
----------------------------------------------------------
    τ_L      -0.084         +0.030            2.1%    7.4%
    τ_G      +0.001         +0.048            1.1%    2.0%
   COMP      -0.003         +0.061            1.3%    2.2%

Expected if each module did its job: τ_G positive on the genre axis,
τ_L negative on the language axis. Expected if each carries its confound:
τ_L negative on the genre axis, τ_G positive on the language axis.


In [31]:
# ── CORE ───────────────────────────────────────────────────────────────
def layer_of_name(n):
    for p in n.split("."):
        if p.isdigit(): return int(p)
    return -1

def clean(tau, axis, label):
    """Returns a copy of tau with its component along `axis` removed from
    o_proj and down_proj — the two matrices writing into the residual stream."""
    out, touched = {}, 0
    for n, (B_, A_, s) in tau.items():
        if "o_proj" in n or "down_proj" in n:
            c = min(layer_of_name(n)+1, NC-1)
            e = axis[c].to(B_.dtype)
            if e.shape[0] == B_.shape[0]:
                B_ = B_ - torch.outer(e, e @ B_); touched += 1
        out[n] = (B_, A_, s)
    print(f"  {label}: component removed from {touched} matrices")
    return out

TAU_L_c = clean(TAU_L, E_GENRE, "τ_L without its GENRE component")
TAU_G_c = clean(TAU_G, E_LANG,  "τ_G without its LANGUAGE component")

REF = f"fitted λ ({LL},{LG})"
TRIALS = [
    (REF,                       [(TAU_L,   LL, None), (TAU_G,   LG, None)]),
    ("+ τ_G cleaned of language",[(TAU_L,   LL, None), (TAU_G_c, LG, None)]),
    ("+ τ_L cleaned of genre",   [(TAU_L_c, LL, None), (TAU_G,   LG, None)]),
    (PRESPECIFIED,               [(TAU_L_c, LL, None), (TAU_G_c, LG, None)]),
]
INT = {}
for name, recipe in TRIALS:
    v = load_and_measure(combine(recipe), ["target·val", "target·TEST"])
    INT[name] = (v["target·val"][0], v["target·TEST"][0])

ref_test = INT[REF][1]
print(f"\n{'system':<30}{'bpb val':>10}{'bpb TEST':>11}{'Δ vs fitted λ':>16}")
print("-"*67)
for name, _ in TRIALS:
    tag = "  ← PRE-SPECIFIED" if name == PRESPECIFIED else ("  (reference)" if name == REF else "")
    print(f"{name:<30}{INT[name][0]:>10.4f}{INT[name][1]:>11.4f}"
          f"{INT[name][1]-ref_test:>+16.4f}{tag}")

d_pre  = INT[PRESPECIFIED][1] - ref_test
others = [INT[n][1]-ref_test for n, _ in TRIALS if n not in (PRESPECIFIED, REF)]
print(f"\nRESULT (the one that goes into the paper)")
print(f"  full intervention, pre-specified: {d_pre:+.4f} bpb over the fitted λ")
print(f"OPTIMISTIC UPPER BOUND (not a measurement)")
print(f"  best of the {len(TRIALS)-1} variants, chosen by looking at TEST: "
      f"{min([d_pre]+others):+.4f} bpb")

for name, recipe in ((REF, TRIALS[0][1]), (PRESPECIFIED, TRIALS[3][1])):
    if name not in NATS: NATS[name] = nats_of_system(combine(recipe))
m, lo, hi = ci(NATS[PRESPECIFIED], NATS[REF], "chapter")
print(f"\ncontrast «{PRESPECIFIED}» − «{REF}»")
print(f"  Δ = {m:+.4f} bpb · CI95 by chapter [{lo:+.4f}, {hi:+.4f}]"
      f"{'   ✱ excludes zero' if not (lo <= 0 <= hi) else '   — includes zero'}")
wins = 0
for c in chapters:
    s = CHAPLAB == c
    d = (NATS[PRESPECIFIED][s].sum()-NATS[REF][s].sum())/math.log(2)/BYTES_BLK[s].sum()
    wins += d < 0
    print(f"    chapter {c}: {d:+.4f}")
print(f"  chapters in favour of the intervention: {wins}/{len(chapters)}")

json.dump({"cos_axes": cos_axes.tolist(),
           "decomposition": {k: {kk: list(map(float, vv)) for kk, vv in v.items()}
                             for k, v in DEC.items()},
           "intervention": {k: list(v) for k, v in INT.items()},
           "prespecified": PRESPECIFIED, "delta_prespecified": float(d_pre),
           "protocol": "axes built on target·val; not zero-shot"},
          open(f"{CKPT}/confound_axes.json", "w"), indent=1, ensure_ascii=False)
print(f"\n→ {CKPT}/confound_axes.json")
del dL, dG, dC, H0, H0f; free()
check_base("confound axes")

  τ_L without its GENRE component: component removed from 48 matrices
  τ_G without its LANGUAGE component: component removed from 48 matrices

system                           bpb val   bpb TEST   Δ vs fitted λ
-------------------------------------------------------------------
fitted λ (0.3,0.3)                1.1215     1.0888         +0.0000  (reference)
+ τ_G cleaned of language         1.1215     1.0888         -0.0001
+ τ_L cleaned of genre            1.1213     1.0887         -0.0001
+ both cleaned                    1.1214     1.0887         -0.0001  ← PRE-SPECIFIED

RESULT (the one that goes into the paper)
  full intervention, pre-specified: -0.0001 bpb over the fitted λ
OPTIMISTIC UPPER BOUND (not a measurement)
  best of the 3 variants, chosen by looking at TEST: -0.0001 bpb

contrast «+ both cleaned» − «fitted λ (0.3,0.3)»
  Δ = -0.0002 bpb · CI95 by chapter [-0.0003, -0.0001]   ✱ excludes zero
    chapter 15: -0.0001
    chapter 16: -0.0001
    chapter 17: -0.0005
    ch

True

## 17 · Export

Everything the paper reports, in one file. The per-cell JSON files written above
hold the detail; this one is the summary.

In [32]:
# ══ PLUMBING ═══════════════════════════════════════════════════════════
#  Infrastructure only: batching, OOM handling, mounting and unmounting
#  adapters, I/O. No decision here is part of the study's argument.
# ═══════════════════════════════════════════════════════════════════════
summary = {
    "versions": VERSIONS,
    "config": {"r": R, "alpha": ALPHA, "lr": LORA_LR, "seq_len": SEQ_LEN,
               "batch": BATCH, "seed": SEED, "backbone": BACKBONE,
               "target": TARGET, "legal_pair": LEGAL_PAIR,
               "trainable_parameters": 14_917_632},
    "steps": STEPS, "bytes": BYTES, "tokens": TOKENS,
    "backbone_before": {k: list(v) for k, v in BEFORE.items()},
    "training_history": {"language": {str(k): v for k, v in HIST_L.items()},
                         "genre":    {str(k): v for k, v in HIST_G.items()}},
    "zero_shot": {label: {k: list(v) for k, v in RES[label].items()}
                  for label, _ in SYSTEMS},
    "subspace_survival": {t_: math.sqrt(v[1]/v[0]) for t_, v in surviving.items()},
    "chapters": chapters, "blocks_per_chapter": {int(c): int((CHAPLAB == c).sum())
                                                 for c in chapters},
    "lambda_grid": {"chosen": [LAM_L, LAM_G],
                    "points": {k: list(v) for k, v in FIT.items()}},
}
with open(f"{CKPT}/elma_sild_sv.json", "w") as f:
    json.dump(summary, f, indent=1, ensure_ascii=False)
print(f"→ {CKPT}/elma_sild_sv.json")
try:
    from google.colab import files; files.download(f"{CKPT}/elma_sild_sv.json")
except Exception as e:
    print("(download manually)", e)

→ /content/drive/MyDrive/elma_sv/elma_sild_sv.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>